In [ ]:
from __future__ import annotations   # must remain the very first statement of the notebook

import copy
import csv
import glob
import itertools
import math
import os
import re
import sys
import time
import traceback
from collections import deque, namedtuple
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, replace
from dataclasses import field as dc_field
from functools import lru_cache
from types import SimpleNamespace

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from scipy import ndimage
from scipy.spatial import cKDTree
from scipy.stats import rankdata

# Constellation Detection as Plate Solving
*CS-GY 6643 Computer Vision, Project 1, Question 5*

## The problem in one paragraph

Every scene gives us a 3000 × 3000 greyscale sky and a folder of 32 × 32 query patches. A patch is one of three things: a star belonging to the constellation drawn in that sky, some other star of the same sky, or a crop taken from somewhere else entirely. Patches that do belong to the sky have been rotated, rescaled, blurred, shifted by a sub-pixel amount and had noise added, but their centre still sits on the answer. For each patch the output is either `-1` or `(x, y, m)`, where `m = 1` marks a star of the figure, and each scene additionally gets one of 48 constellation names.

A scene is scored as

$$0.25\,\text{Presence} + 0.20\,\text{Localisation} + 0.25\,\text{GeometricRecovery} + 0.30\,\text{Identification},$$

and the final score is the mean over scenes.

## How the solver is organised

The code is built around a handful of objects, each owning one stage of a scene:

| object | job |
|---|---|
| `SkyModel` | background removal, matched-filter star catalogue, saturated-star fixes, stretched correlation image, neighbour tables |
| `Patch` | the same front-end at patch scale, plus the neighbour descriptor and the halo size |
| `Constellation` | a reference drawing turned into a normalised point set with seed pairs |
| `PoseScorer` | correlation of one patch against the sky at any set of (centre, zoom, angle) poses |
| `Localizer` | finds every patch of a scene: whole-sky sweep → screen → coarse → fine, or the bright-star path |
| `Identifier` | exhaustive two-point hypotheses for every figure, likelihood-ratio scoring, scrambled-figure calibration |
| `SceneOutcome` | everything a solved scene produced (answers, name, ranking, the evidence behind them) |

The notebook follows the data: environment and settings, then the stages in pipeline order, then the scene driver, diagnostics and runs.

## 0. Environment

Locates the `participant/` folder (Colab Drive, a local folder, or the `CONST_DATA` variable) and picks the torch device. `IN_NOTEBOOK` gates everything that only makes sense interactively (plots, evaluation), so the same file can also run as a script.

In [ ]:
def running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


def find_participant_dir():
    """Absolute path of the first candidate folder that holds a patterns/ subfolder, else None."""
    candidates = (os.environ.get("CONST_DATA", ""), "participant", "../participant",
                  os.path.join(os.getcwd(), "participant"),
                  "/content/drive/MyDrive/constellation_data/participant", "/content/participant")
    return next((os.path.abspath(c) for c in candidates if c and os.path.isdir(os.path.join(c, "patterns"))), None)


IN_COLAB = running_in_colab()
DATA = find_participant_dir()
if DATA is None and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = find_participant_dir()
if DATA is None:
    print("participant/ not found: place it next to this notebook, set CONST_DATA, or pass --data")

IN_NOTEBOOK = ("ipykernel" in sys.modules) or hasattr(sys, "ps1") or IN_COLAB
print("data:", DATA, "| colab:", IN_COLAB, "| notebook:", IN_NOTEBOOK)

SKY_W = SKY_H = 3000
CV_THREADS = int(os.environ.get("DENSE_WORKERS", "8"))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, os.cpu_count() or 1))
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

## 1. What the data looks like

- **Scenes.** Three labelled training skies (pisces 41 patches, scorpius 41, taurus 34) and sixteen validation skies with between 21 and 87 patches.
- **Reference drawings.** RGBA line art. Size, rotation, mirroring and aspect ratio are all arbitrary, so a drawing only fixes the figure up to a similarity (or mild affine) transform.
- **Patches.** Present, off-figure and foreign patches are degraded in exactly the same way. Their appearance alone says nothing about their class; only a match against the sky can.
- **Sky statistics.** The intensity distribution is heavy-tailed: most pixels are near black, a small fraction is saturated, and the background shifts across mosaic seams and nebulae. Hence a robust background model, and a compressive stretch before any correlation so a single saturated star cannot dominate a window.

No part of the solver may be tuned to an individual scene. Every constant is expressed in pixels, robust-noise units or score units, and the training scenes serve only as a check.

## 2. Configuration

All knobs sit in the `Config` dataclass. `configure(base, **changes)` returns an edited copy; the experiments at the end of the notebook use it to try variants without touching the defaults.

In [ ]:
@dataclass
class Config:
    """Tunable numbers of the solver.  Each one is a pixel, noise-unit or score quantity; none
    is tied to a particular scene."""

    # --- sky catalogue ---------------------------------------------------------------
    bg_shrink: int = 4                 # the background is estimated on a 4x-downsampled sky
    bg_median_px: int = 31             # median window on that small image (~124 px at full size)
    star_psf_sigma: float = 1.2        # width of the point-source matched filter
    star_k_sigma: float = 3.6          # detection threshold in robust-noise units
    star_nms_radius: int = 3
    star_cap: int = 60000
    despeckle: bool = True             # 3x3 median, detection only
    sat_value: int = 250               # saturation level
    sat_min_px: int = 3                # a plateau must have at least this many pixels
    sat_min_fill: float = 0.3          # and be compact (seams and diffraction spikes are not)

    # --- query patches ---------------------------------------------------------------
    tile_bg_median: int = 9
    tile_bg_model: str = "median"      # "median" or "surface"
    tile_edge_margin: float = 0.0
    tile_psf_sigma: float = 1.0
    tile_k_sigma: float = 3.2
    tile_remove_halo: bool = True
    tile_max_neighbours: int = 8
    tile_seed_neighbours: int = 3
    anchor_radius: float = 2.5
    tile_centre: float = 15.5

    # --- neighbour-pattern voting ------------------------------------------------------
    zoom_lo: float = 0.7               # sky pixels per patch pixel
    zoom_hi: float = 1.45
    sky_knn: int = 16
    nb_tol_px: float = 1.5
    nb_tol_rel: float = 0.06
    nb_tol_edge: float = 2.5
    flux_spread_w: float = 0.3
    mem_budget_gb: float = 20.0

    # --- whole-sky template sweep -----------------------------------------------------
    search_stride: int = 1
    search_n_angles: int = 36
    search_zooms: tuple = (0.8, 0.9, 1.0, 1.12, 1.25)
    search_inner_r: float = 0.0
    search_peaks_each: int = 6
    search_keep: int = 1500
    search_batch: int = 24
    search_backend: str = "auto"       # "auto" | "cv" | "torch"

    # --- verification correlation ------------------------------------------------------
    corr_sigma: float = 1.0
    corr_stretch: str = "sqrt"         # "asinh" | "sqrt" | "none"
    corr_outer_r: float = 22.0
    corr_remove_profile: bool = True
    corr_inner_r: float = 4.0
    corr_batch: int = 65536
    screen_keep: int = 150
    coarse_dang_deg: float = 7.5
    coarse_dzoom: float = 0.10
    fine_keep: int = 12
    fine_dang_deg: float = 1.5
    fine_dzoom: float = 0.025
    fine_dxy: float = 0.6
    null_draws: int = 200
    bgcorr_star_level: float = 3.0
    bgcorr_inner_r: float = 7.0
    bgcorr_weight: float = 0.0
    whiten: bool = True
    whiten_eps: float = 0.1
    whiten_dense: bool = False

    # --- presence decision -------------------------------------------------------------
    pres_ncc_alone: float = 0.70
    pres_ncc_floor: float = 0.55
    pres_margin: float = 0.08
    pres_bright_ncc: float = 0.55
    pres_z_alone: float = 8.0
    pres_z_weak: float = 3.0
    pres_min_support: int = 3
    pres_zbg: float = 99.0

    # --- reference drawings ------------------------------------------------------------
    fig_min_pair_len: float = 0.35
    fig_pair_cap: int = 500
    fig_pairs_per_node: int = 4

    # --- identification ----------------------------------------------------------------
    id_n_stars: int = 350
    id_n_complete: int = 600
    id_scale_lo: float = 300.0         # sky pixels per normalised figure unit
    id_scale_hi: float = 1500.0
    id_shift_margin: float = 300.0
    id_verify_min: int = 60
    id_verify_max: int = 400
    id_keep_per_chunk: int = 400
    id_chunk_work: float = 2.0e7
    id_tol_rel: float = 0.02
    id_tol_lo: float = 10.0
    id_tol_hi: float = 12.0
    id_seed_mult: float = 2.5
    id_soft_mult: float = 1.5
    id_dt_cell: float = 4.0
    id_sigma_div: float = 3.0
    id_affine_min: int = 4
    id_max_aniso: float = 1.3
    id_off_margin: float = 80.0
    id_off_penalty: float = 0.0
    id_max_off: int = 1
    id_max_off_frac: float = 0.4
    id_size_w: float = 0.4
    id_missing_w: float = 0.5
    id_offimg_w: float = 0.1
    id_merge_px: float = 6.0
    null_k: int = 2
    null_sd_floor: float = 1.0
    null_seed: int = 12345
    null_shrink: float = 2.0
    null_bw: float = 1.5

    # --- bright patches and figure placement ---------------------------------------------
    bright_radius: float = 6.0
    bright_pool: int = 500
    bright_refine_n: int = 60
    bright_jitter_n: int = 6
    bright_jitter_px: float = 2.0
    bright_jitter_step: float = 1.0
    bright_jitter_sat_px: float = 6.0
    bright_jitter_sat_step: float = 2.0
    core_search_r: float = 8.0
    core_level: float = 0.9
    bright_always_present: bool = True
    bright_sure_margin: float = 0.08
    dense_confident_margin: float = 0.10
    node_snap_px: float = 30.0
    node_n_angles: int = 18
    node_zooms: tuple = (0.8, 0.9, 1.0, 1.12, 1.25)
    node_min_ncc: float = 0.0
    node_min_margin: float = 0.0
    node_keep_px: float = 12.0
    place_tol_mult: float = 3.0
    blown_mean: float = 150.0
    assign_blown: bool = True
    blown_flux_frac: float = 0.5
    member_px: float = 12.0

    # --- execution -----------------------------------------------------------------------
    prefetch: int = 3
    do_identify: bool = True
    verbose: int = 1


def configure(base: Config | None = None, **changes) -> Config:
    """Copy of `base` (or of the defaults) with the given fields replaced."""
    return replace(base or Config(), **changes)

## 3. Shared utilities

- `robust_sigma`: Gaussian-equivalent noise from the median absolute deviation.
- `greedy_suppress`: keep the strongest point of every cluster (non-maximum suppression on points).
- `fit_similarity` / `fit_affine`: least-squares 2-D transforms between matched point sets.
- `tile_geometry` and `ring_labels`: cached pixel offsets, radii and one-pixel ring indices of a 32 × 32 patch.
- `StarTable`: a column store of detections whose columns are always filtered together.

In [ ]:
def log(cfg, *parts):
    if cfg.verbose:
        print(*parts, file=sys.stderr, flush=True)


def robust_sigma(values) -> float:
    """1.4826 x median absolute deviation; very large inputs are thinned to ~400k samples."""
    sample = np.asarray(values, np.float32).ravel()
    if sample.size > 400000:
        sample = sample[:: int(sample.size // 400000) + 1]
    spread = np.abs(sample - np.median(sample))
    return float(1.4826 * np.median(spread) + 1e-6)


def greedy_suppress(points, radius, strength):
    """Boolean mask: visiting points strongest first, keep a point unless an already-kept point
    lies within `radius` of it."""
    keep = np.zeros(len(points), bool)
    if len(points) == 0:
        return keep
    neighbourhoods = cKDTree(points).query_ball_point(points, radius)
    alive = np.ones(len(points), bool)
    for i in np.argsort(-strength):
        if alive[i]:
            keep[i] = True
            alive[neighbourhoods[i]] = False
    return keep


def fit_similarity(src, dst, flip):
    """Least squares dst ~ A src + t with A = scale * rotation * diag(1, flip) (Umeyama)."""
    mirror = np.array([1.0, float(flip)])
    mirrored = src * mirror
    src_mean, dst_mean = mirrored.mean(0), dst.mean(0)
    a, b = mirrored - src_mean, dst - dst_mean
    U, sv, Vt = np.linalg.svd(a.T @ b)
    D = np.diag([1.0, np.sign(np.linalg.det(Vt.T @ U.T)) or 1.0])
    rotation = Vt.T @ D @ U.T
    scale = (sv * np.diag(D)).sum() / ((a ** 2).sum() + 1e-12)
    A = scale * rotation @ np.diag(mirror)
    return A, dst_mean - A @ src.mean(0)


def fit_affine(src, dst):
    homogeneous = np.hstack([src, np.ones((len(src), 1))])
    M = np.linalg.lstsq(homogeneous, dst, rcond=None)[0]
    return M[:2].T, M[2]


def apply_transform(A, t, pts):
    return pts @ A.T + t


@lru_cache(maxsize=None)
def tile_geometry(c=15.5):
    """(dx, dy, radius) of every pixel of a 32 x 32 patch, measured from the patch centre."""
    offsets = np.arange(32, dtype=np.float32) - c
    dy, dx = np.meshgrid(offsets, offsets, indexing="ij")
    return dx, dy, np.hypot(dx, dy)


@lru_cache(maxsize=None)
def ring_labels(c=15.5):
    """Integer ring index (rounded radius, capped at 23) of every patch pixel."""
    return np.minimum(np.round(tile_geometry(c)[2]).astype(int), 23)


def pixel_of(coord, limit):
    return np.clip(np.round(coord).astype(int), 0, limit - 1)


@dataclass
class StarTable:
    """Detections as parallel columns.  Optional columns are None until filled."""
    x: np.ndarray
    y: np.ndarray
    flux: np.ndarray
    peak: np.ndarray
    snr: np.ndarray = None
    x0: np.ndarray = None
    y0: np.ndarray = None
    sat_area: np.ndarray = None
    sat_flux: np.ndarray = None
    sat: np.ndarray = None

    @classmethod
    def empty(cls):
        return cls(*(np.zeros(0) for _ in range(4)))

    def __len__(self):
        return len(self.x)

    @property
    def xy(self):
        return np.stack([self.x, self.y], 1)

    def subset(self, selector) -> "StarTable":
        return StarTable(**{name: (col if col is None else col[selector]) for name, col in vars(self).items()})

## 4. From pixels to stars

### 4a. The sky: `SkyModel`

1. **Flatten.** Downsample 4× by area averaging, take a 31-pixel median, upsample, subtract. Stars are sparse bright outliers and the background is roughly piecewise constant, which is exactly the situation a median handles well.
2. **Detect.** A 3 × 3 median removes hot pixels (for detection only). A σ = 1.2 px Gaussian is the matched filter for a point source; its local maxima above 3.6 × (1.4826 · MAD) are stars. Their centroids and fluxes come from 5 × 5 box sums.
3. **Fix saturated stars.** A blown-out star has a flat top, so the box centroid can land anywhere on it. Each such star is moved to the centroid of its compact saturated component (found through a 3 × 3 maximum filter over the component labels), duplicates on one plateau are merged, and the count of saturated pixels in a 25 × 25 box (read from a summed-area table) becomes its brightness proxy.
4. **Prepare for correlation.** The flattened sky is lightly smoothed and mapped through `sign(x)·sqrt(|x|)` in noise units: brightness order is preserved but the bright tail is compressed.
5. **Neighbour tables.** Once the patches are known (they fix the search radius), every star records its 16 nearest neighbours as offsets, distances, angles and fluxes.

In [ ]:
def subtract_background(img8, cfg):
    """Sky minus its median-filtered thumbnail, scaled back to full size."""
    H, W = img8.shape
    thumb = cv2.resize(img8, (W // cfg.bg_shrink, H // cfg.bg_shrink), interpolation=cv2.INTER_AREA)
    smooth = cv2.medianBlur(thumb, cfg.bg_median_px | 1)
    background = cv2.resize(smooth, (W, H), interpolation=cv2.INTER_LINEAR).astype(np.float32)
    return img8.astype(np.float32) - background


def detect_peaks(response, signal, threshold, radius, cap, border=2, win=5) -> StarTable:
    """Local maxima of `response` above `threshold`, excluding a `border`-pixel frame, with
    intensity-weighted centroids and fluxes from win x win box sums of `signal`."""
    side = 2 * radius + 1
    accept = (response >= cv2.dilate(response, np.ones((side, side), np.uint8))) & (response > threshold)
    if border > 0:
        frame = np.ones(accept.shape, bool)
        frame[border:-border, border:-border] = False
        accept[frame] = False
    rows, cols = np.nonzero(accept)
    if rows.size == 0:
        return StarTable.empty()
    weight = np.maximum(signal, 0).astype(np.float32)
    H, W = signal.shape
    col_coord = np.broadcast_to(np.arange(W, dtype=np.float32), (H, W))
    row_coord = np.broadcast_to(np.arange(H, dtype=np.float32)[:, None], (H, W))
    box_at_peaks = lambda img: cv2.boxFilter(img, -1, (win, win), normalize=False)[rows, cols]
    mass = box_at_peaks(weight) + 1e-6
    cx = box_at_peaks(weight * col_coord) / mass
    cy = box_at_peaks(weight * row_coord) / mass
    strength = response[rows, cols]
    flat_top = (np.abs(cx - cols) > win) | (np.abs(cy - rows) > win)     # plateau: fall back to the pixel
    cx = np.where(flat_top, cols.astype(np.float32), cx)
    cy = np.where(flat_top, rows.astype(np.float32), cy)
    centres = np.stack([cx, cy], 1)
    survivors = greedy_suppress(centres, radius, strength)
    centres, mass, strength = centres[survivors], mass[survivors], strength[survivors]
    order = np.argsort(-strength)[:cap]
    return StarTable(centres[order, 0], centres[order, 1], mass[order], strength[order])


def box_count(mask, size):
    """Number of set pixels in the size x size box centred on each pixel, borders mirrored
    (reflect-101), computed from a summed-area table."""
    r = size // 2
    table = cv2.integral(cv2.copyMakeBorder(mask.astype(np.uint8), r, r, r, r, cv2.BORDER_REFLECT_101))
    return table[size:, size:] - table[:-size, size:] - table[size:, :-size] + table[:-size, :-size]


def recentre_saturated(img8, hp, stars: StarTable, cfg) -> StarTable:
    """Move saturated stars to the centroid of their compact saturated plateau and keep one
    detection per plateau.  Fills x0/y0 (original position), sat_area and sat_flux."""
    n = len(stars)
    stars.x0, stars.y0 = stars.x.copy(), stars.y.copy()
    stars.sat_area = np.zeros(n, np.float32)
    stars.sat_flux = np.zeros(n, np.float32)
    saturated = img8 >= cfg.sat_value
    if n == 0 or cfg.sat_value <= 0 or not saturated.any():
        return stars
    _, labels, stats, centroids = cv2.connectedComponentsWithStats(saturated.astype(np.uint8), connectivity=8)
    H, W = img8.shape
    r, c = pixel_of(stars.y, H), pixel_of(stars.x, W)
    own = labels[r, c]
    nearby = ndimage.maximum_filter(labels, size=3, mode="constant", cval=0)[r, c]
    region = np.where(own > 0, own, nearby)
    area = stats[region, cv2.CC_STAT_AREA]
    fill = area / np.maximum(stats[region, cv2.CC_STAT_WIDTH] * stats[region, cv2.CC_STAT_HEIGHT], 1)
    hits = np.flatnonzero((region > 0) & (area >= cfg.sat_min_px) & (fill >= cfg.sat_min_fill))
    if hits.size == 0:
        return stars
    plateau = region[hits]
    stars.x[hits], stars.y[hits] = centroids[plateau, 0], centroids[plateau, 1]
    stars.sat_area[hits] = area[hits]
    stars.sat_flux[hits] = ndimage.sum_labels(np.maximum(hp, 0).astype(np.float64), labels, plateau)
    _, first = np.unique(plateau, return_index=True)        # earliest detection on each plateau wins
    duplicate = np.zeros(n, bool)
    duplicate[hits] = True
    duplicate[hits[first]] = False
    return stars.subset(~duplicate)


def compress_range(img, mode="asinh"):
    """Express in robust-noise units, then squash the bright tail."""
    scaled = img / robust_sigma(img if img.size < 400000 else img[::7, ::7])
    squash = {"asinh": np.arcsinh, "sqrt": lambda v: np.sign(v) * np.sqrt(np.abs(v))}.get(mode, lambda v: v)
    return squash(scaled).astype(np.float32)


class SkyModel:
    """Star catalogue, correlation image and noise level of one sky; neighbour tables are added
    by index_neighbours once the search radius is known."""

    def __init__(self, img8, cfg):
        H, W = img8.shape
        hp = subtract_background(img8, cfg)
        detection = cv2.medianBlur(hp, 3) if cfg.despeckle else hp
        response = cv2.GaussianBlur(detection, (0, 0), cfg.star_psf_sigma)
        self.noise = robust_sigma(response[::7, ::7])
        stars = detect_peaks(response, detection, cfg.star_k_sigma * self.noise, cfg.star_nms_radius, cfg.star_cap, border=3)
        stars = recentre_saturated(img8, hp, stars, cfg)
        stars.sat = box_count(img8 >= 250, 25)[pixel_of(stars.y, H), pixel_of(stars.x, W)].astype(np.float32)
        self.stars = stars
        self.corr_image = compress_range(cv2.GaussianBlur(hp, (0, 0), cfg.corr_sigma), cfg.corr_stretch)
        self.xy = np.stack([stars.x, stars.y], 1).astype(np.float32)
        self.flux = stars.flux.astype(np.float32)
        self.sat = stars.sat.astype(np.float32)
        self.sat_area = np.asarray(stars.sat_area, np.float32)
        self.N = len(self.xy)
        self.tree = cKDTree(self.xy)

    def index_neighbours(self, cfg, R):
        dist, nb = self.tree.query(self.xy, k=cfg.sky_knn + 1, distance_upper_bound=R)
        dist, nb = dist[:, 1:], nb[:, 1:]                      # drop the star itself
        valid = np.isfinite(dist)
        nb = np.where(valid, nb, 0)
        offset = self.xy[nb] - self.xy[:, None, :]
        dx = np.where(valid, offset[..., 0], 1e6).astype(np.float32)
        dy = np.where(valid, offset[..., 1], 1e6).astype(np.float32)
        self.nb_idx, self.nb_valid = nb, valid
        self.nb_dx, self.nb_dy = dx, dy
        self.nb_r = np.where(valid, np.hypot(dx, dy), np.nan).astype(np.float32)
        self.nb_phi = np.arctan2(dy, dx).astype(np.float32)
        self.nb_flux = np.where(valid, self.flux[nb], np.nan).astype(np.float32)
        return self

    def star_list(self):
        """The light-weight part of the catalogue that outlives the scene."""
        return SimpleNamespace(xy=self.xy, flux=self.flux, sat=self.sat)

### 4b. The patches: `Patch`

Each patch goes through a scaled-down version of the same pipeline. A small median followed by a Gaussian models the local illumination. The mean of each one-pixel ring around the centre is then subtracted, so the halo of a bright central star does not bury its neighbours. The detection within 2.5 px of the centre is the *anchor*; up to 8 other detections, with their distance, angle, flux and an SNR weight, form the patch descriptor.

The *halo radius* is where the ring profile falls below 25 % of the central excess. A patch whose halo reaches 6 px is handled as a **bright-star patch** later on.

In [ ]:
def surface_background(tile):
    """Quadratic surface through the non-star pixels of a patch (4 rounds of 2-sigma clipping)."""
    h, w = tile.shape
    yy, xx = np.mgrid[0:h, 0:w]
    x, y = xx.ravel(), yy.ravel()
    design = np.stack([np.ones(h * w), x, y, x ** 2, y ** 2, (xx * yy).ravel()], 1) / 32.0
    values = tile.ravel()
    inliers = np.ones(h * w, bool)
    for _ in range(4):
        model = design @ np.linalg.lstsq(design[inliers], values[inliers], rcond=None)[0]
        resid = values - model
        inliers = resid < 2.0 * robust_sigma(resid[inliers])
    return model.reshape(h, w).astype(np.float32)


def ring_average(img, ring):
    return np.bincount(ring.ravel(), img.ravel(), 24) / np.maximum(np.bincount(ring.ravel(), minlength=24), 1)


def halo_radius(img_c, cfg):
    """First radius (3..11 px) at which the ring-mean profile falls below 25 % of the central excess."""
    rad = tile_geometry(cfg.tile_centre)[2]
    profile = np.array([img_c[(rad >= max(k - 0.5, 0.0)) & (rad < k + 0.5)].mean() for k in range(16)])
    floor = np.median(profile[12:16])
    limit = 0.25 * max(np.nanmax(profile[0:3]) - floor, 1e-3)
    fading = [profile[k] - floor < limit for k in range(3, 12)]
    steps = fading.index(True) if True in fading else len(fading)
    return min(3.0 + steps, 11.0)


class Patch:
    """A query patch: its stretched image, neighbour descriptor and halo size."""

    def __init__(self, t8, cfg):
        self.raw = t8
        self.mean = float(t8.mean())
        hp = self._background_free(t8, cfg)
        self.img_c = compress_range(cv2.GaussianBlur(hp, (0, 0), cfg.corr_sigma), cfg.corr_stretch)
        self._describe(self._find_stars(hp, cfg), cfg, t8.shape[0])
        self.halo_r = halo_radius(self.img_c, cfg)

    @staticmethod
    def _background_free(t8, cfg):
        tile = t8.astype(np.float32)
        if cfg.tile_bg_model == "surface":
            background = surface_background(tile)
        else:
            background = cv2.GaussianBlur(cv2.medianBlur(t8, cfg.tile_bg_median | 1).astype(np.float32), (0, 0), 3)
        return cv2.medianBlur(tile - background, 3)

    @staticmethod
    def _find_stars(hp, cfg) -> StarTable:
        signal = hp
        if cfg.tile_remove_halo:
            ring = ring_labels(cfg.tile_centre)
            signal = hp - ring_average(hp, ring)[ring]
        response = cv2.GaussianBlur(signal, (0, 0), cfg.tile_psf_sigma)
        noise = robust_sigma(response)
        stars = detect_peaks(response, signal, cfg.tile_k_sigma * noise, 2, 24, border=0)
        stars.snr = stars.peak / noise
        lo, hi = cfg.tile_edge_margin, hp.shape[0] - 1 - cfg.tile_edge_margin
        return stars.subset((lo <= stars.x) & (stars.x <= hi) & (lo <= stars.y) & (stars.y <= hi))

    def _describe(self, stars: StarTable, cfg, size):
        c = cfg.tile_centre
        xy = stars.xy if len(stars) else np.zeros((0, 2))
        from_centre = np.hypot(xy[:, 0] - c, xy[:, 1] - c)
        anchor = int(np.argmin(from_centre)) if len(xy) and from_centre.min() <= cfg.anchor_radius else -1
        self.has_anchor = anchor >= 0
        if self.has_anchor:
            ax, ay = xy[anchor, 0], xy[anchor, 1]
            self.core_flux = float(stars.flux[anchor])
        else:
            ax = ay = c
            self.core_flux = float(np.maximum(self.img_c[14:18, 14:18], 0).sum()) + 1.0
        rest = np.arange(len(xy)) != anchor
        dx, dy = xy[rest, 0] - ax, xy[rest, 1] - ay
        dist = np.hypot(dx, dy)
        pick = np.flatnonzero(dist >= 2.0)
        pick = pick[np.argsort(-stars.flux[rest][pick])[: cfg.tile_max_neighbours]]   # brightest first
        px, py = xy[rest, 0][pick], xy[rest, 1][pick]
        snr = stars.snr[rest][pick]
        self.nb_r = dist[pick].astype(np.float32)
        self.nb_theta = np.arctan2(dy[pick], dx[pick]).astype(np.float32)
        self.nb_flux = stars.flux[rest][pick].astype(np.float32)
        self.nb_edge = (np.minimum(px, py) < 2) | (np.maximum(px, py) > size - 3)
        weight = 0.4 + 0.6 * np.minimum(1.0, snr / 10.0)
        self.nb_weight = np.where(self.nb_edge, 0.6 * weight, weight).astype(np.float32)


def core_centroid(raw, cfg):
    """Weighted centroid (patch px) and pixel count of the central star core: the connected
    region above core_level of the peak excess on the lightly blurred raw patch, so that a
    saturated plateau survives."""
    img = cv2.GaussianBlur(np.asarray(raw, np.float32), (0, 0), 1.0)
    U, V, rad = tile_geometry(cfg.tile_centre)
    floor = float(np.median(img[rad >= 12]))
    iy, ix = np.unravel_index(int(np.argmax(np.where(rad <= cfg.core_search_r, img, -np.inf))), img.shape)
    top = float(img[iy, ix])
    if not np.isfinite(top) or top - floor <= 1e-3:
        return cfg.tile_centre, cfg.tile_centre, 0
    thr = floor + cfg.core_level * (top - floor)
    _, lab, _, _ = cv2.connectedComponentsWithStats((img >= thr).astype(np.uint8), connectivity=8)
    core = lab == lab[iy, ix]
    w = np.maximum(img - thr, 0)[core] + 1e-3
    return (float((w * U[core]).sum() / w.sum()) + cfg.tile_centre,
            float((w * V[core]).sum() / w.sum()) + cfg.tile_centre, int(core.sum()))

## 5. Reference figures: `Constellation`

A drawing's stars are its white dots: pixels with alpha above 100 whose three colour channels all exceed 150. Every connected blob of 4 to 400 px (and at most 30 px across) becomes a node, with its area kept as a size, since bigger dots mean brighter stars. A blob that is far larger than the typical one, or clearly elongated, is really two dots touching; it is split at the maxima of its distance transform. Drawings without white dots fall back to filled discs, rings and skeleton end points.

Nodes are normalised to zero centroid and unit RMS radius. Hypotheses are later seeded from node pairs; to bound the cost (pairs × nodes), each node contributes at most its 4 longest pairs.

In [ ]:
def drawing_mask(im):
    """Foreground of a drawing: the alpha channel when it is informative, otherwise pixels far
    from the most common border grey level."""
    gray = im if im.ndim == 2 else cv2.cvtColor(im[:, :, :3], cv2.COLOR_BGR2GRAY)
    if im.ndim == 3 and im.shape[2] == 4:
        alpha = im[:, :, 3]
        if alpha.min() < 128 < alpha.max():
            return (alpha > 128).astype(np.uint8)
    border = np.concatenate([gray[0], gray[-1], gray[:, 0], gray[:, -1]])
    background = int(np.bincount(border).argmax())
    return (np.abs(gray.astype(int) - background) > 40).astype(np.uint8)


def seed_pairs(norm, cfg):
    """Node pairs of normalised length >= fig_min_pair_len (or the upper 70 % if that leaves
    fewer than 3), capped at fig_pairs_per_node x nodes, longest first."""
    pairs = np.array(list(itertools.combinations(range(len(norm)), 2)), int).reshape(-1, 2)
    if len(pairs) == 0:
        return np.zeros((0, 2), int)
    length = np.hypot(*(norm[pairs[:, 0]] - norm[pairs[:, 1]]).T)
    long_enough = length >= cfg.fig_min_pair_len
    if long_enough.sum() < 3:
        long_enough = length >= np.percentile(length, 30)
    pairs, length = pairs[long_enough], length[long_enough]
    cap = min(cfg.fig_pair_cap, max(12, cfg.fig_pairs_per_node * len(norm)))
    if len(pairs) > cap:
        pairs = pairs[np.argsort(-length)[:cap]]
    return pairs if len(pairs) else np.zeros((0, 2), int)


def split_touching(labels, i, x, y, w, h):
    """Centres of the dots inside blob i (distance-transform maxima), or None for a single dot."""
    blob = np.pad((labels[y:y + h, x:x + w] == i).astype(np.uint8), 1)
    dt = cv2.distanceTransform(blob, cv2.DIST_L2, 5)
    ridge = (dt >= cv2.dilate(dt, np.ones((3, 3), np.uint8))) & (dt >= 0.6 * dt.max()) & (dt > 0)
    rows, cols = np.nonzero(ridge)
    if len(rows) < 2:
        return None
    centres = np.stack([cols, rows], 1).astype(float)
    keep = greedy_suppress(centres, max(4.0, 1.2 * float(dt.max())), dt[rows, cols])
    if keep.sum() < 2:
        return None
    return centres[keep] - 1.0 + np.array([x, y], float)


def white_dot_nodes(im):
    """(centres, areas) of the drawing's white dots, touching pairs split in two."""
    alpha = im[:, :, 3] if im.shape[2] == 4 else np.full(im.shape[:2], 255, np.uint8)
    dots = (alpha > 100) & (im[:, :, :3].min(axis=2) > 150)
    count, labels, stats, centres = cv2.connectedComponentsWithStats(dots.astype(np.uint8))
    ids = np.arange(1, count)
    size_ok = (stats[ids, cv2.CC_STAT_AREA] >= 4) & (stats[ids, cv2.CC_STAT_AREA] <= 400)
    span_ok = np.maximum(stats[ids, cv2.CC_STAT_WIDTH], stats[ids, cv2.CC_STAT_HEIGHT]) <= 30
    ids = ids[size_ok & span_ok]
    typical = float(np.median(stats[ids, cv2.CC_STAT_AREA])) if len(ids) else 0.0
    nodes, sizes = [], []
    for i in ids:
        x, y, w, h, area = stats[i]
        suspicious = area > 1.6 * typical or max(w, h) / max(1.0, float(min(w, h))) > 1.35
        parts = split_touching(labels, i, x, y, w, h) if suspicious else None
        if parts is None:
            nodes.append(centres[i])
            sizes.append(area)
        else:
            nodes.extend(parts)
            sizes.extend([area / len(parts)] * len(parts))
    return nodes, sizes


def outline_nodes(mask):
    """Fallback for drawings without white dots: filled discs, rings, then skeleton ends and
    junctions.  Returns (nodes, sizes, line width)."""
    from skimage.morphology import skeletonize
    H, W = mask.shape
    nodes, sizes = [], []
    dt = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
    skel = skeletonize(mask.astype(bool)) if mask.any() else None
    has_skel = skel is not None and skel.any()
    lw = max(float(2 * np.median(dt[skel])) if has_skel else 2.0, 1.0)
    r = int(max(2, round(0.8 * lw + 1)))
    disc = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))
    n, _, st, ce = cv2.connectedComponentsWithStats(cv2.morphologyEx(mask, cv2.MORPH_OPEN, disc))
    for i in range(1, n):
        x, y, w, h, area = st[i]
        if max(w, h) <= 6 * lw + 10 and area >= 0.45 * w * h:
            nodes.append(ce[i])
            sizes.append(area)
    n, _, st, ce = cv2.connectedComponentsWithStats((1 - mask).astype(np.uint8))
    for i in range(1, n):
        x, y, w, h, area = st[i]
        enclosed = x > 0 and y > 0 and x + w < W and y + h < H
        if enclosed and max(w, h) <= 4 * lw + 8 and area >= 0.5 * w * h:
            nodes.append(ce[i])
            sizes.append(area + math.pi * lw * (math.sqrt(area / math.pi) + lw / 2))
    if len(nodes) < 3 and has_skel:
        sk = skel.astype(np.uint8)
        degree = cv2.filter2D(sk, -1, np.ones((3, 3), np.uint8), borderType=cv2.BORDER_CONSTANT) - sk
        ends = np.argwhere((sk > 0) & ((degree == 1) | (degree >= 3)))[:, ::-1].astype(float)
        if len(ends):
            for q in ends[greedy_suppress(ends, 2 * lw + 2, np.ones(len(ends)))]:
                nodes.append(q)
                sizes.append(lw * lw)
    return nodes, sizes, lw


@dataclass
class Constellation:
    name: str
    raw: np.ndarray       # node positions in drawing pixels
    nodes: np.ndarray     # zero centroid, unit RMS radius
    sizes: np.ndarray     # dot areas
    pairs: np.ndarray     # node pairs that seed hypotheses
    shape: tuple

    @classmethod
    def from_png(cls, path, cfg) -> "Constellation":
        im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        mask = drawing_mask(im)
        nodes, sizes = white_dot_nodes(im) if im.ndim == 3 else ([], [])
        lw = 2.0
        if len(nodes) < 2:
            nodes, sizes, lw = outline_nodes(mask)
        nodes, sizes = np.array(nodes, float).reshape(-1, 2), np.array(sizes, float)
        if len(nodes):
            keep = greedy_suppress(nodes, 1.5 * lw + 1, sizes)
            nodes, sizes = nodes[keep], sizes[keep]
        norm = nodes.copy()
        if len(nodes) >= 2:
            centred = nodes - nodes.mean(0)
            norm = centred / (math.sqrt((centred ** 2).sum(1).mean()) + 1e-9)
        name = re.sub(r"(_pattern)?\.png$", "", os.path.basename(path))
        return cls(name, nodes, norm, sizes, seed_pairs(norm, cfg), mask.shape)

    def scrambled(self, rng, cfg) -> "Constellation":
        """Null figure: every node rotated about the centroid by its own random angle."""
        angle = rng.uniform(0, 2 * math.pi, len(self.nodes))
        radius = np.hypot(self.nodes[:, 0], self.nodes[:, 1])
        norm = np.stack([radius * np.cos(angle), radius * np.sin(angle)], 1)
        return Constellation(self.name + "~", self.raw, norm, self.sizes, seed_pairs(norm, cfg), self.shape)


def load_patterns(root, cfg):
    patterns = []
    for path in sorted(glob.glob(os.path.join(root, "patterns", "*.png"))):
        patterns.append(Constellation.from_png(path, cfg))
        pat = patterns[-1]
        log(cfg, f"  figure {pat.name:18s} nodes={len(pat.nodes):3d} pairs={len(pat.pairs):4d}")
    return patterns

## 6. Comparing a patch with the sky

- `SkyTensors` mirrors a `SkyModel` on the torch device.
- `neighbour_votes` is a cheap geometric pre-filter: for every (patch, sky star) pair it counts how many patch neighbours land on sky neighbours under the rotation and zoom implied by one of the patch's brightest neighbours.
- `PoseScorer` prepares one patch once (masked, ring-flattened, optionally ring-whitened, zero-mean template) and then scores it at any batch of (centre, zoom, angle) poses by bilinear resampling of the sky. Removing the azimuthal mean profile from *both* sides means a round halo correlates with nothing; only the asymmetric surroundings carry signal. `best_nearby` evaluates a local grid of angle × zoom × shift offsets around each pose.
- Two interchangeable **sweepers** run the whole-sky template search. `TorchSweep` uses a batched `conv2d` on the GPU. `OpenCVSweep` uses `cv2.matchTemplate` (`TM_CCORR`) for the numerator, divided by the local standard deviation under the same disc mask. Both give the same correlation maps.

In [ ]:
class SkyTensors:
    """Device copy of a SkyModel: correlation image, star positions and neighbour tables."""

    def __init__(self, sky: SkyModel, dev):
        put = lambda a: torch.as_tensor(a, device=dev)
        self.dev, self.N = dev, sky.N
        self.xy, self.flux, self.sat_area = put(sky.xy), put(sky.flux), put(sky.sat_area)
        self.nb_dx, self.nb_dy = put(sky.nb_dx), put(sky.nb_dy)
        self.nb_r = put(np.nan_to_num(sky.nb_r, nan=1e6))
        self.nb_phi, self.nb_valid = put(sky.nb_phi), put(sky.nb_valid)
        self.nb_flux = put(np.nan_to_num(sky.nb_flux, nan=1.0))
        self.img = put(sky.corr_image)[None, None]
        self.H, self.W = sky.corr_image.shape


Votes = namedtuple("Votes", "count score zoom angle")


def neighbour_votes(patches, st: SkyTensors, cfg) -> Votes:
    """For every (patch, sky star): the best neighbour agreement over the patch's first
    tile_seed_neighbours neighbours used as the rotation/zoom seed."""
    dev = st.dev
    B, K_t = len(patches), cfg.tile_max_neighbours

    def stacked(attr, fill):
        table = np.full((B, K_t), fill, np.float32)
        for b, patch in enumerate(patches):
            values = getattr(patch, attr)
            table[b, :len(values)] = values
        return torch.as_tensor(table, device=dev)

    r_t, th_t, w_t = stacked("nb_r", 1.0), stacked("nb_theta", 0.0), stacked("nb_weight", 0.0)
    f_t, e_t = stacked("nb_flux", 1.0), stacked("nb_edge", 0.0)
    n_t = torch.as_tensor(np.array([len(p.nb_r) for p in patches], np.int64), device=dev)
    N, K = st.nb_dx.shape
    best = Votes(torch.zeros((B, N), dtype=torch.int32, device=dev), torch.full((B, N), -1.0, device=dev),
                 torch.ones((B, N), device=dev), torch.zeros((B, N), device=dev))
    real = torch.arange(K_t, device=dev)[None, :] < n_t[:, None]
    n_seed = min(cfg.tile_seed_neighbours, int(n_t.max().item()) if B else 0)
    bytes_per_patch = N * K * K_t * K * 4 * 6
    budget = cfg.mem_budget_gb * 1e9 if dev.type == "cuda" else 1.5e9
    group = max(1, int(budget // bytes_per_patch))
    sky_dx = st.nb_dx[None, :, None, None, :]
    sky_dy = st.nb_dy[None, :, None, None, :]
    zero, minus_one = torch.zeros((), device=dev), torch.full((), -1.0, device=dev)
    big, small = torch.full((), -1e9, device=dev), torch.full((), 1e9, device=dev)

    for start in range(0, B, group):
        sl = slice(start, min(B, start + group))
        b = sl.stop - sl.start
        for seed in range(n_seed):
            has_seed = n_t[sl] > seed
            if not bool(has_seed.any()):
                continue
            # rotation/zoom that maps the seed neighbour onto each sky neighbour
            s = st.nb_r[None] / r_t[sl, seed, None, None]
            ok = st.nb_valid[None] & (s >= cfg.zoom_lo) & (s <= cfg.zoom_hi) & has_seed[:, None, None]
            a = st.nb_phi[None] - th_t[sl, seed, None, None]
            # every patch neighbour under that pose, compared with every sky neighbour
            ang = a[..., None] + th_t[sl, None, None, :]
            rr = s[..., None] * r_t[sl, None, None, :]
            px, py = rr * torch.cos(ang), rr * torch.sin(ang)
            d2 = torch.square(px[..., None] - sky_dx)
            d2.add_(torch.square(py[..., None] - sky_dy))
            tol = cfg.nb_tol_px + cfg.nb_tol_rel * rr + cfg.nb_tol_edge * e_t[sl, None, None, :]
            hit = (d2 <= (tol * tol)[..., None]) & real[sl, None, None, :, None]
            tile_hit, sky_hit = hit.any(-1), hit.any(-2)
            count = torch.minimum(tile_hit.sum(-1), sky_hit.sum(-1))
            dmin, nearest = d2.min(-1)
            del d2, hit
            # geometric quality, SNR-weighted, minus a penalty for inconsistent flux ratios
            quality = torch.where(tile_hit, 1.0 - 0.7 * torch.sqrt(dmin) / tol, zero)
            weighted = (quality * w_t[sl, None, None, :]).sum(-1)
            f_sky = st.nb_flux[None, :, None, :].expand(b, N, K, K).gather(3, nearest)
            log_ratio = torch.log(f_t[sl, None, None, :] + 1e-6) - torch.log(f_sky + 1e-6)
            hi = torch.where(tile_hit, log_ratio, big).amax(-1)
            lo = torch.where(tile_hit, log_ratio, small).amin(-1)
            spread = torch.where(tile_hit.any(-1), (hi - lo).clamp(0, 3.0), zero)
            score = torch.where(ok, 0.6 * count.float() + weighted - cfg.flux_spread_w * spread, minus_one)
            top, j = score.max(-1)
            at_best = lambda t: t.gather(2, j[..., None])[..., 0]
            better = top > best.score[sl]
            best.score[sl] = torch.where(better, top, best.score[sl])
            best.count[sl] = torch.where(better, at_best(count).int(), best.count[sl])
            best.zoom[sl] = torch.where(better, at_best(s), best.zoom[sl])
            best.angle[sl] = torch.where(better, at_best(a), best.angle[sl])
    return best


@lru_cache(maxsize=None)
def device_offsets(dev_name, step, c):
    dx, dy, rad = tile_geometry(c)
    return tuple(torch.as_tensor(a[::step, ::step], device=torch.device(dev_name)) for a in (dx, dy, rad))


Pose = namedtuple("Pose", "score ncc bgc nbg zoom angle x y")
NO_SHIFT = np.zeros((1, 2))


class PoseScorer:
    """Correlates one patch with the sky at many poses.  The template side (mask, ring
    flattening, optional ring whitening, zero mean) is prepared once in the constructor."""

    N_RING = 24

    def __init__(self, st: SkyTensors, tile, cfg, step=1, inner=None, whiten=None, bg_inner=0.0):
        self.st, self.cfg, self.dev = st, cfg, st.dev
        self.U, self.V, rad = device_offsets(str(st.dev), step, cfg.tile_centre)
        star_px = F.max_pool2d((tile.abs() > cfg.bgcorr_star_level).float()[None, None], 3, 1, 1)[0, 0] > 0
        if step > 1:
            tile, star_px = tile[::step, ::step], star_px[::step, ::step]
        self.tile, self.L = tile, self.U.shape[0]
        self.whiten = cfg.whiten if whiten is None else whiten
        inner = cfg.corr_inner_r if inner is None else inner
        self.mask = ((rad <= cfg.corr_outer_r) & (rad >= inner)).float()
        self.mask_flat = self.mask.reshape(-1)
        self.ring = torch.clamp(rad.round().long(), max=23).reshape(-1)
        self.ring_n = torch.zeros(self.N_RING, device=self.dev).index_add_(0, self.ring, self.mask_flat) + 1e-6
        shaped = self._condition((tile * self.mask)[None], gate=None)[0]
        self.T = (shaped - (shaped * self.mask).sum() / self.mask.sum()) * self.mask
        self.T_norm = torch.sqrt((self.T * self.T).sum()) + 1e-6
        self.bg_mask = (self.mask > 0) & (~star_px) & (rad >= max(cfg.bgcorr_inner_r, bg_inner))
        self.batch = cfg.corr_batch if step > 1 else cfg.corr_batch // 4
        self.x_max, self.y_max = float(st.W - 1), float(st.H - 1)

    def _ring_sums(self, X):
        return torch.zeros(X.shape[0], self.N_RING, device=self.dev).index_add_(1, self.ring, X)

    def _condition(self, X, gate):
        """Remove the ring means and optionally scale each ring to unit RMS; X is (B, L, L)."""
        B, L = X.shape[0], self.L
        if self.cfg.corr_remove_profile:
            flat = X.reshape(B, -1)
            X = ((flat - (self._ring_sums(flat) / self.ring_n)[:, self.ring]) * self.mask_flat).reshape(B, L, L)
            X = X if gate is None else X * gate
        if self.whiten:
            flat = X.reshape(B, -1)
            rms = torch.sqrt(self._ring_sums(flat * flat) / self.ring_n) + self.cfg.whiten_eps
            X = (flat / rms[:, self.ring] * self.mask_flat).reshape(B, L, L)
            X = X if gate is None else X * gate
        return X

    def _sample(self, cx, cy, zoom, ang):
        co, sn, z = torch.cos(ang)[:, None, None], torch.sin(ang)[:, None, None], zoom[:, None, None]
        X = cx[:, None, None] + z * (co * self.U - sn * self.V)
        Y = cy[:, None, None] + z * (sn * self.U + co * self.V)
        m = len(cx)
        grid = torch.stack([X * (2.0 / self.x_max) - 1.0, Y * (2.0 / self.y_max) - 1.0], -1).reshape(1, m * self.L, self.L, 2)
        window = F.grid_sample(self.st.img, grid, mode="bilinear", padding_mode="zeros", align_corners=True)
        valid = ((X >= 0) & (X <= self.x_max) & (Y >= 0) & (Y <= self.y_max) & (self.mask > 0)).float()
        return window.reshape(m, self.L, self.L), valid

    def _background(self, window, shaped, valid):
        """Correlation restricted to star-free pixels outside the halo, and its pixel count."""
        sky_px = F.max_pool2d((window.abs() > self.cfg.bgcorr_star_level).float()[:, None], 3, 1, 1)[:, 0] > 0
        bm = ((valid > 0) & self.bg_mask[None] & (~sky_px)).float()
        count = bm.sum((1, 2)) + 1e-6
        centre = lambda X: (X - (X.sum((1, 2)) / count)[:, None, None]) * bm
        a, b = centre(shaped * bm), centre(self.tile[None] * bm)
        return (a * b).sum((1, 2)) / (torch.sqrt((a * a).sum((1, 2)) * (b * b).sum((1, 2))) + 1e-6), count

    def __call__(self, cx, cy, zoom, ang, background=True):
        """(ncc, background ncc, background pixel count) for each pose."""
        n, dev = len(cx), self.dev
        if n == 0:
            nothing = torch.zeros(0, device=dev)
            return nothing, nothing, nothing
        ncc, bgc, nbg = torch.empty(n, device=dev), torch.empty(n, device=dev), torch.empty(n, device=dev)
        for lo in range(0, n, self.batch):
            hi = min(n, lo + self.batch)
            window, valid = self._sample(cx[lo:hi], cy[lo:hi], zoom[lo:hi], ang[lo:hi])
            count = valid.sum((1, 2)) + 1e-6
            shaped = self._condition(window * valid, gate=valid)
            centred = (shaped - (shaped.sum((1, 2)) / count)[:, None, None]) * valid
            ncc[lo:hi] = (centred * self.T[None]).sum((1, 2)) / (torch.sqrt((centred * centred).sum((1, 2))) * self.T_norm + 1e-6)
            if background:
                bgc[lo:hi], nbg[lo:hi] = self._background(window, shaped, valid)
        return ncc, bgc, nbg

    def ncc(self, cx, cy, zoom, ang):
        return self(cx, cy, zoom, ang, background=False)[0]

    def best_nearby(self, cx, cy, zoom, ang, d_ang, d_zoom, d_xy) -> Pose:
        """Best of the angle x zoom x shift grid around every pose."""
        dev = self.dev
        f32 = lambda a: torch.as_tensor(a, device=dev, dtype=torch.float32)
        d_ang, d_zoom, d_xy = f32(d_ang), f32(d_zoom), f32(d_xy)
        ia, iz, ix = (idx.ravel() for idx in torch.meshgrid(
            torch.arange(len(d_ang), device=dev), torch.arange(len(d_zoom), device=dev),
            torch.arange(len(d_xy), device=dev), indexing="ij"))
        n, g = len(cx), len(ia)
        AA = (ang[:, None] + d_ang[ia][None, :]).ravel()
        ZZ = (zoom[:, None] * d_zoom[iz][None, :]).ravel()
        XX = (cx[:, None] + d_xy[ix, 0][None, :]).ravel()
        YY = (cy[:, None] + d_xy[ix, 1][None, :]).ravel()
        v, bg, nb = (t.view(n, g) for t in self(XX, YY, ZZ, AA))
        score = v + self.cfg.bgcorr_weight * bg
        k, rows = score.argmax(1), torch.arange(n, device=dev)
        pick = lambda t: t.view(n, g)[rows, k]
        return Pose(pick(score), pick(v), pick(bg), pick(nb), pick(ZZ), pick(AA), pick(XX), pick(YY))


def angle_offsets(half_deg, steps):
    return np.deg2rad(half_deg) * np.array(steps)


def subpixel_shifts(cfg):
    sub = cfg.fine_dxy * np.array([-2, -1, 0, 1, 2]) / 2.0
    return np.array([(x, y) for x in sub for y in sub])


def square_shifts(half_width, step):
    g = np.arange(-half_width, half_width + 1e-6, max(step, 0.25)) if half_width > 0 else np.zeros(1)
    return np.array([(x, y) for x in g for y in g])

In [ ]:
def template_bank(tile, cfg, dev):
    """The patch resampled at every (angle, zoom) as zero-mean, unit-norm kernels in search-image
    pixels (angle-major order), plus one disc mask per zoom."""
    stride, n_ang = cfg.search_stride, cfg.search_n_angles
    angles = np.arange(n_ang) * (2 * np.pi / n_ang)
    zooms = np.array(cfg.search_zooms, np.float32)
    r = int(math.ceil(cfg.corr_outer_r * zooms.max() / stride))
    u = torch.arange(-r, r + 1, device=dev, dtype=torch.float32) * stride
    X, Y = torch.meshgrid(u, u, indexing="xy")
    zoom_t = torch.as_tensor(zooms, device=dev)
    a = torch.as_tensor(angles, device=dev, dtype=torch.float32).repeat_interleave(len(zooms))
    z = zoom_t.repeat(n_ang)
    ca, sa, zz = torch.cos(a)[:, None, None], torch.sin(a)[:, None, None], z[:, None, None]
    px = (ca * X + sa * Y) / zz + cfg.tile_centre
    py = (-sa * X + ca * Y) / zz + cfg.tile_centre
    grid = torch.stack([px / 31.0 * 2 - 1, py / 31.0 * 2 - 1], -1)
    kernels = F.grid_sample(tile[None, None].expand(len(a), 1, 32, 32), grid, mode="bilinear",
                            padding_mode="zeros", align_corners=True)[:, 0]
    disc = torch.hypot(X, Y)[None] / zoom_t[:, None, None]
    masks = ((disc >= cfg.search_inner_r) & (disc <= cfg.corr_outer_r)).float()
    support = masks.repeat(n_ang, 1, 1)
    kernels = (kernels - (kernels * support).sum((1, 2))[:, None, None] / support.sum((1, 2))[:, None, None]) * support
    kernels = kernels / (torch.sqrt((kernels * kernels).sum((1, 2)))[:, None, None] + 1e-6)
    return kernels[:, None], masks[:, None], a, z


class TorchSweep:
    """Whole-sky sweep as a batched conv2d; the local std maps are computed once per scene."""

    def __init__(self, st: SkyTensors, cfg):
        self.cfg, self.dev, self.stride = cfg, st.dev, cfg.search_stride
        self.img = F.avg_pool2d(st.img, self.stride) if self.stride > 1 else st.img
        self.img2 = self.img * self.img
        self._std = {}

    def _local_std(self, masks):
        key = tuple(masks.shape)
        if key not in self._std:
            r, area = masks.shape[-1] // 2, masks.sum((1, 2, 3))
            s1, s2 = F.conv2d(self.img, masks, padding=r), F.conv2d(self.img2, masks, padding=r)
            self._std[key] = torch.sqrt((s2 - s1 * s1 / area[None, :, None, None]).clamp_min(1e-3))
        return self._std[key]

    def sweep(self, tile):
        """(x, y, zoom, angle, ncc) of the strongest peaks over all templates."""
        cfg, dev = self.cfg, self.dev
        bank, masks, angles, zooms = template_bank(tile, cfg, dev)
        std = self._local_std(masks)
        n_zoom, r, width, k = len(cfg.search_zooms), bank.shape[-1] // 2, self.img.shape[-1], cfg.search_peaks_each
        chunks = []
        for t0 in range(0, len(bank), cfg.search_batch):
            t1 = min(len(bank), t0 + cfg.search_batch)
            ncc = F.conv2d(self.img, bank[t0:t1], padding=r)[0] / std[0, torch.arange(t0, t1, device=dev) % n_zoom]
            ncc = torch.where(ncc >= F.max_pool2d(ncc[:, None], 5, 1, 2)[:, 0], ncc, torch.full((), -1.0, device=dev))
            v, idx = torch.topk(ncc.reshape(t1 - t0, -1), k, dim=1)
            chunks.append((v.ravel(), torch.div(idx, width, rounding_mode="floor").ravel(), (idx % width).ravel(),
                           angles[t0:t1].repeat_interleave(k), zooms[t0:t1].repeat_interleave(k)))
        val, ys, xs, ang, zm = (torch.cat(col) for col in zip(*chunks))
        keep = torch.topk(val, min(cfg.search_keep, len(val))).indices
        half = (self.stride - 1) / 2.0
        return xs[keep].float() * self.stride + half, ys[keep].float() * self.stride + half, zm[keep], ang[keep], val[keep]


class OpenCVSweep:
    """CPU twin of TorchSweep: TM_CCORR numerator over the masked window std, templates spread
    over a thread pool."""

    def __init__(self, st: SkyTensors, cfg):
        self.cfg, self.dev, self.stride = cfg, st.dev, cfg.search_stride
        img = F.avg_pool2d(st.img, self.stride) if self.stride > 1 else st.img
        self.img = np.ascontiguousarray(img[0, 0].cpu().numpy().astype(np.float32))
        self._std = {}

    def _local_std(self, masks, r):
        key = (masks.shape, r)
        if key not in self._std:
            padded = cv2.copyMakeBorder(self.img, r, r, r, r, cv2.BORDER_CONSTANT, value=0.0)
            squared = padded * padded
            maps = []
            for m in masks:
                m = np.ascontiguousarray(m.astype(np.float32))
                s1 = cv2.matchTemplate(padded, m, cv2.TM_CCORR)
                var = cv2.matchTemplate(squared, m, cv2.TM_CCORR) - s1 * s1 / float(m.sum())
                np.maximum(var, 1e-3, out=var)
                maps.append(np.sqrt(var, dtype=np.float32))
            self._std[key] = (padded, maps)
        return self._std[key]

    def sweep(self, tile):
        cfg, dev = self.cfg, self.dev
        bank, masks, angles, zooms = template_bank(tile, cfg, dev)
        bank = bank[:, 0].cpu().numpy().astype(np.float32)
        masks = masks[:, 0].cpu().numpy().astype(np.float32)
        n_zoom, r = len(cfg.search_zooms), bank.shape[-1] // 2
        padded, std_maps = self._local_std(masks, r)
        width, k = self.img.shape[1], cfg.search_peaks_each
        footprint = np.ones((5, 5), np.uint8)

        def strongest(i):
            ncc = cv2.matchTemplate(padded, np.ascontiguousarray(bank[i]), cv2.TM_CCORR) / std_maps[i % n_zoom]
            idx = np.flatnonzero(ncc >= cv2.dilate(ncc, footprint))
            val = ncc.ravel()[idx]
            if len(idx) > k:
                sel = np.argpartition(-val, k - 1)[:k]
                idx, val = idx[sel], val[sel]
            tag = lambda v: np.full(len(idx), float(v), np.float32)
            return val.astype(np.float32), idx // width, idx % width, tag(angles[i]), tag(zooms[i])

        with ThreadPoolExecutor(max_workers=CV_THREADS) as pool:
            val, ys, xs, ang, zm = (np.concatenate(col) for col in zip(*pool.map(strongest, range(len(bank)))))
        n = min(cfg.search_keep, len(val))
        keep = np.argpartition(-val, n - 1)[:n] if len(val) > n else np.arange(len(val))
        half = (self.stride - 1) / 2.0
        as_t = lambda a: torch.as_tensor(a, device=dev)
        return (as_t(xs[keep].astype(np.float32) * self.stride + half), as_t(ys[keep].astype(np.float32) * self.stride + half),
                as_t(zm[keep]), as_t(ang[keep]), as_t(val[keep]))


def make_sweeper(st, cfg):
    """OpenCV on CPU (or when asked for), torch conv2d otherwise."""
    use_cv = DEVICE.type == "cpu" or getattr(cfg, "search_backend", "auto") == "cv"
    return (OpenCVSweep if use_cv else TorchSweep)(st, cfg)

## 7. Locating the patches: `Localizer`

**Ordinary patches** go through a funnel:

| step | candidates | what is evaluated |
|---|---|---|
| whole-sky sweep | 36 angles × 5 zooms, 5 × 5 max-pool peaks, best 1,500 kept | template correlation |
| neighbour votes | sky stars with ≥ 2 agreeing neighbours are added | geometry only |
| screen | best 150 | halo-free correlation at half resolution |
| coarse grid | best 12 | ±7.5° × ±10 % zoom |
| fine grid | winner | ±1.5° × ±2.5 % zoom × sub-pixel shifts |

For each patch the solver records its best correlation, the best competitor more than 5 px away, and a z-score against 200 random poses on random stars.

**Bright-star patches** are different: in the whole-sky sweep their halo matches every bright star equally well. Instead, the patch's own core centroid is placed on each of the 500 brightest catalogue stars, and the pose is scored with the full-disc, ring-whitened correlation so that the faint neighbours decide. The best 60 are refined on a coarse grid and the best 6 get a shift search (wider on saturated stars) and a sub-pixel polish.

In [ ]:
@dataclass
class TileResult:
    x: float = -1.0
    y: float = -1.0
    ncc: float = -1.0
    ncc_alt: float = -1.0       # best competitor more than 5 px away
    z: float = 0.0              # (ncc - null median) / null MAD
    bgc: float = 0.0
    z_bgc: float = 0.0
    n_bgc: float = 0.0
    support: int = 0            # neighbour votes; negative means "placed on a figure node"
    n_nb: int = 0
    zoom: float = 1.0
    present: bool = False
    m: int = 0
    star: int = -1
    has_anchor: bool = False
    halo_r: float = 0.0
    mean: float = 0.0
    via_bright: bool = False
    n_pool: int = 0
    pool_margin: float = 0.0
    on_saturated: bool = False
    core_dx: float = 0.0
    core_dy: float = 0.0
    core_px: int = 0


def robust_z(value, sample):
    centre = float(sample.median())
    spread = float(1.4826 * (sample - centre).abs().median()) + 1e-3
    return (value - centre) / spread


@dataclass
class PoseBook:
    """Current best pose per candidate star (all device tensors)."""
    v: torch.Tensor
    x: torch.Tensor
    y: torch.Tensor
    s: torch.Tensor
    a: torch.Tensor
    bg: torch.Tensor
    nbg: torch.Tensor
    refined: torch.Tensor

    def write(self, idx, pose: Pose):
        self.v[idx], self.bg[idx], self.nbg[idx] = pose.ncc, pose.bgc, pose.nbg
        self.s[idx], self.a[idx], self.x[idx], self.y[idx] = pose.zoom, pose.angle, pose.x, pose.y


def scan_star_list(scorer: PoseScorer, star_xy, star_sat, core, cfg, refine_n=None) -> PoseBook:
    """Put the patch core on every listed star over an angle x zoom grid; refine the best
    `refine_n` on a coarse grid, then give the best few a shift search (wider on saturated
    stars) followed by a sub-pixel polish."""
    dev = scorer.dev
    n, n_ang = len(star_xy), cfg.node_n_angles
    zooms = torch.tensor(cfg.node_zooms, device=dev, dtype=torch.float32)
    per = n_ang * len(zooms)
    base = torch.arange(n_ang, device=dev, dtype=torch.float32) * (2 * math.pi / n_ang)
    A, Z = base.repeat_interleave(len(zooms)).repeat(n), zooms.repeat(n_ang * n)
    SX, SY = star_xy[:, 0].repeat_interleave(per), star_xy[:, 1].repeat_interleave(per)
    du, dv = float(core[0]) - cfg.tile_centre, float(core[1]) - cfg.tile_centre
    ca, sa = torch.cos(A), torch.sin(A)
    CX, CY = SX - Z * (ca * du - sa * dv), SY - Z * (sa * du + ca * dv)
    best, where = scorer.ncc(CX, CY, Z, A).view(n, per).max(1)
    rows = torch.arange(n, device=dev)
    at_best = lambda t: t.view(n, per)[rows, where].clone()
    book = PoseBook(best.clone(), at_best(CX), at_best(CY), at_best(Z), at_best(A),
                    torch.zeros(n, device=dev), torch.zeros(n, device=dev), torch.zeros(n, dtype=torch.bool, device=dev))
    M = n if refine_n is None else min(int(refine_n), n)
    if M <= 0:
        return book
    top = torch.topk(best, M).indices
    book.write(top, scorer.best_nearby(book.x[top], book.y[top], book.s[top], book.a[top],
                                       angle_offsets(360.0 / n_ang / 2, [-1, -0.5, 0, 0.5, 1]),
                                       1.0 + 0.1 * np.array([-1, 0, 1]), NO_SHIFT))
    book.refined[top] = True
    fine_a = angle_offsets(cfg.fine_dang_deg, [-1, 0, 1])
    fine_z = 1.0 + cfg.fine_dzoom * np.array([-1, 0, 1])
    finalists = top[torch.topk(book.v[top], min(int(cfg.bright_jitter_n), M)).indices]
    saturated = torch.as_tensor(np.asarray(star_sat, bool), device=dev)
    for group, half, step in ((finalists[saturated[finalists]], cfg.bright_jitter_sat_px, cfg.bright_jitter_sat_step),
                              (finalists[~saturated[finalists]], cfg.bright_jitter_px, cfg.bright_jitter_step)):
        if len(group):
            wide = scorer.best_nearby(book.x[group], book.y[group], book.s[group], book.a[group],
                                      fine_a, fine_z, square_shifts(half, step))
            book.write(group, scorer.best_nearby(wide.x, wide.y, wide.zoom, wide.angle, fine_a, fine_z, subpixel_shifts(cfg)))
    return book


class Localizer:
    """Finds every patch of one scene in its sky.  One torch generator (seed 0) drives the
    random-pose nulls of all patches in order."""

    def __init__(self, patches, st: SkyTensors, cfg):
        self.patches, self.st, self.cfg = patches, st, cfg
        self.sweeper = make_sweeper(st, cfg)
        self.votes = neighbour_votes(patches, st, cfg)
        self.gen = torch.Generator()
        self.gen.manual_seed(0)

    def run(self):
        return [self.locate(k) for k in range(len(self.patches))]

    def locate(self, k) -> TileResult:
        patch, cfg = self.patches[k], self.cfg
        tile = torch.as_tensor(patch.img_c, device=DEVICE)
        res = TileResult(n_nb=len(patch.nb_r), has_anchor=patch.has_anchor, halo_r=float(patch.halo_r), mean=float(patch.mean))
        if self.st.N == 0:
            return res
        votes = (self.votes.count[k], self.votes.zoom[k], self.votes.angle[k])
        if patch.halo_r >= cfg.bright_radius and cfg.bright_pool > 0:
            return self._on_bright_stars(patch, tile, votes, res)
        return self._by_sweep(patch, tile, votes, res)

    def _attach_null(self, scorer, res):
        st, gen = self.st, self.gen
        k = min(self.cfg.null_draws, st.N)
        stars = torch.randperm(st.N, generator=gen)[:k].to(st.dev)
        ang = torch.rand(k, generator=gen).to(st.dev) * 2 * math.pi
        zoom = torch.exp(torch.rand(k, generator=gen).to(st.dev) * (math.log(1.25) - math.log(0.8)) + math.log(0.8))
        ncc, bgc, _ = scorer(st.xy[stars, 0], st.xy[stars, 1], zoom, ang)
        res.z, res.z_bgc = robust_z(res.ncc, ncc), robust_z(res.bgc, bgc)

    def _candidates(self, patch, tile, votes):
        """Sweep peaks, plus the sky stars whose neighbour tables agree with the patch."""
        cx, cy, Z, A, _ = self.sweeper.sweep(tile)
        n_nb = len(patch.nb_r)
        if n_nb >= 1:
            count, vz, va = votes
            agreeing = torch.nonzero(count >= (2 if n_nb >= 2 else 1))[:, 0]
            if len(agreeing):
                cx, cy = torch.cat([cx, self.st.xy[agreeing, 0]]), torch.cat([cy, self.st.xy[agreeing, 1]])
                Z, A = torch.cat([Z, vz[agreeing]]), torch.cat([A, va[agreeing]])
        return cx, cy, Z, A

    def _by_sweep(self, patch, tile, votes, res):
        cfg, st = self.cfg, self.st
        inner, wh = max(cfg.corr_inner_r, patch.halo_r), cfg.whiten_dense
        cx, cy, Z, A = self._candidates(patch, tile, votes)
        # 1. screen at half resolution
        v = PoseScorer(st, tile, cfg, step=2, inner=inner, whiten=wh).ncc(cx, cy, Z, A)
        keep = torch.topk(v, min(cfg.screen_keep, len(v))).indices
        # 2. coarse angle/zoom grid, 3. fine angle/zoom/sub-pixel grid on the best few
        scorer = PoseScorer(st, tile, cfg, inner=inner, whiten=wh, bg_inner=patch.halo_r)
        coarse = scorer.best_nearby(cx[keep], cy[keep], Z[keep], A[keep], angle_offsets(cfg.coarse_dang_deg, [-1, -0.5, 0, 0.5, 1]),
                                    1.0 + cfg.coarse_dzoom * np.array([-1, 0, 1]), NO_SHIFT)
        keep = torch.topk(coarse.score, min(cfg.fine_keep, len(coarse.score))).indices
        fine = scorer.best_nearby(coarse.x[keep], coarse.y[keep], coarse.zoom[keep], coarse.angle[keep],
                                  angle_offsets(cfg.fine_dang_deg, [-1, 0, 1]), 1.0 + cfg.fine_dzoom * np.array([-1, 0, 1]),
                                  subpixel_shifts(cfg))
        b = int(fine.score.argmax())
        res.x, res.y = float(fine.x[b]), float(fine.y[b])
        res.ncc, res.bgc, res.n_bgc, res.zoom = float(fine.ncc[b]), float(fine.bgc[b]), float(fine.nbg[b]), float(fine.zoom[b])
        rivals = torch.hypot(fine.x - fine.x[b], fine.y - fine.y[b]) > 5.0
        res.ncc_alt = float(fine.ncc[rivals].max()) if bool(rivals.any()) else float(fine.ncc.median())
        d2 = (st.xy[:, 0] - res.x) ** 2 + (st.xy[:, 1] - res.y) ** 2
        res.star = int(d2.argmin())
        if float(d2[res.star]) <= 9.0:
            res.support = int(votes[0][res.star])
        self._attach_null(scorer, res)
        return res

    def _on_bright_stars(self, patch, tile, votes, res):
        cfg, st = self.cfg, self.st
        k = min(cfg.bright_pool, st.N)
        pool = torch.topk(st.flux, k).indices
        core = core_centroid(patch.raw, cfg)
        res.core_dx, res.core_dy, res.core_px = core[0] - cfg.tile_centre, core[1] - cfg.tile_centre, int(core[2])
        sat = (st.sat_area[pool] > 0).cpu().numpy()
        scorer = PoseScorer(st, tile, cfg, inner=0.0, whiten=cfg.whiten, bg_inner=patch.halo_r)
        book = scan_star_list(scorer, st.xy[pool], sat, core, cfg, refine_n=cfg.bright_refine_n)
        b = int(book.v.argmax())
        res.x, res.y = float(book.x[b]), float(book.y[b])
        res.ncc, res.bgc, res.n_bgc, res.zoom = float(book.v[b]), float(book.bg[b]), float(book.nbg[b]), float(book.s[b])
        res.star, res.on_saturated, res.via_bright, res.n_pool = int(pool[b]), bool(sat[b]), True, int(k)
        rivals = torch.hypot(book.x - res.x, book.y - res.y) > 5.0
        res.ncc_alt = float(book.v[rivals].max()) if bool(rivals.any()) else float(book.v.median())
        res.pool_margin = res.ncc - res.ncc_alt
        res.support = int(votes[0][pool[b]])
        self._attach_null(scorer, res)
        return res

## 8. Is the patch in the sky?

A located patch is declared present as soon as any rule in `PRESENCE_RULES` holds:

1. it has already been placed on a figure node;
2. it is a bright-star patch (correlation cannot tell present from absent here, and most such patches are present, so this is a prior);
3. its correlation is at least 0.70;
4. its correlation is at least 0.55 **and** it leads the runner-up by at least 0.08, an additive form of Lowe's ratio test;
5. its z-score is at least 8, or at least 3 when backed by at least 3 neighbour votes (a double threshold: strong evidence alone, weak evidence only with support).

In [ ]:
PRESENCE_RULES = (
    ("placed on a figure node", lambda r, c: r.support < 0),
    ("bright-star prior",       lambda r, c: c.bright_always_present and r.halo_r >= c.bright_radius),
    ("bright and correlated",   lambda r, c: r.halo_r >= c.bright_radius and r.ncc >= c.pres_bright_ncc),
    ("background z-score",      lambda r, c: r.z_bgc >= c.pres_zbg and r.n_bgc >= 60),
    ("strong correlation",      lambda r, c: r.ncc >= c.pres_ncc_alone),
    ("clear lead",              lambda r, c: r.ncc >= c.pres_ncc_floor and (r.ncc - r.ncc_alt) >= c.pres_margin),
    ("far above null",          lambda r, c: r.z >= c.pres_z_alone),
    ("above null with votes",   lambda r, c: r.support >= c.pres_min_support and r.z >= c.pres_z_weak),
)


def decide_presence(r: TileResult, cfg) -> bool:
    return r.x >= 0 and any(rule(r, cfg) for _, rule in PRESENCE_RULES)

## 9. Which constellation is it? `Identifier`

**Point set.** The 350 brightest catalogue stars together with the positions of patches judged present, merged within 6 px. Most of these points are clutter.

**Exhaustive two-point hypotheses.** Two correspondences determine a similarity transform. Every seed node pair is matched against every ordered pair of points in both handednesses; rather than sampling as RANSAC would, the whole set is screened in bulk on the device. A distance raster of the point set gives each projected node's distance `d` to its nearest point, and the node contributes `exp(-½ (d / 1.5·tol)²)`, which rewards tight fits over ones that merely land near stars. The ordered point pairs and the raster depend only on the scene, so the `Identifier` builds them once and shares them across all 48 figures and their nulls.

**Verification.** Between 60 and 400 distinct top hypotheses are matched one-to-one within a 10 to 12 px tolerance and refitted. An affine refit is allowed once there are enough inliers, provided it stays nearly isotropic and does not drift in scale.

**Likelihood ratio.** Each inlier contributes `g = log(area / (m·2πσ²))` minus its squared-residual term, and the fit's free parameters are paid for by subtracting their worth in inliers. In-image nodes with none of the 600 brightest stars nearby cost up to `0.5·g`, off-image nodes cost `0.1·g`, and a positive rank correlation between dot size and star brightness earns a bonus.

**Null calibration.** Small figures fit random stars easily. So each figure is also fitted, on the same points, in two scrambled versions (every node rotated about the centroid by its own random angle), and the reported score is `max(raw, 0) − null mean − null sd`, with the null pooled over figures of similar node count.

In [ ]:
@dataclass
class FigureFit:
    name: str = "unknown"
    score: float = -1e9
    n_in: int = 0
    n_nodes: int = 0
    A: np.ndarray = None
    t: np.ndarray = None
    in_nodes: np.ndarray = None
    in_pts: np.ndarray = None
    tol: float = 0.0
    resid: float = 0.0
    raw: float = 0.0          # max(LLR, 0) before calibration
    null_mean: float = 0.0
    null_sd: float = 0.0
    n_checked: int = 0


def match_nodes(A, t, pattern, Q, tree, tol):
    """Project the nodes and pair them one-to-one with points of Q within tol, closest first.
    Returns (node indices, point indices, distances, projected nodes)."""
    proj = apply_transform(A, t, pattern.nodes)
    dist, nearest = tree.query(proj)
    order = np.flatnonzero(dist <= tol)
    order = order[np.argsort(dist[order])]
    claimed, nodes_in, pts_in = set(), [], []
    for i in order:
        if nearest[i] not in claimed:
            claimed.add(nearest[i])
            nodes_in.append(i)
            pts_in.append(nearest[i])
    nodes_in, pts_in = np.array(nodes_in, int), np.array(pts_in, int)
    return nodes_in, pts_in, (dist[nodes_in] if len(nodes_in) else np.zeros(0)), proj


def calibrate(fits, null_scores, patterns, cfg):
    """score = raw - (shrunk, node-count pooled) null mean - (pooled) null sd."""
    n_nodes = np.array([len(p.nodes) for p in patterns], float)
    own = np.array([np.mean(v) for v in null_scores])
    values = np.concatenate([np.asarray(v, float) for v in null_scores])
    for i, fit in enumerate(fits):
        closeness = np.exp(-0.5 * ((n_nodes - n_nodes[i]) / cfg.null_bw) ** 2)
        weights = np.concatenate([np.full(len(v), closeness[j]) for j, v in enumerate(null_scores)])
        total = max(weights.sum(), 1e-9)
        mu = float((weights * values).sum() / total)
        sd = math.sqrt(float((weights * (values - mu) ** 2).sum() / total))
        fit.null_mean = (cfg.null_k * own[i] + cfg.null_shrink * mu) / (cfg.null_k + cfg.null_shrink)
        fit.null_sd = sd + cfg.null_sd_floor
        fit.score = fit.raw - fit.null_mean - fit.null_sd


class Identifier:
    """Fits reference figures to one scene's point set Q."""

    def __init__(self, Q, qmag, cfg, dev, bright_xy=None):
        self.Q, self.qmag, self.cfg, self.dev = Q, qmag, cfg, dev
        self.tree = cKDTree(Q)
        self.bright_tree = cKDTree(bright_xy) if bright_xy is not None and len(bright_xy) else cKDTree(Q)
        # distance-to-nearest-point raster, with one extra "outside" cell far from everything
        self.cell = cfg.id_dt_cell
        self.G = int(math.ceil(SKY_W / self.cell)) + 2
        empty = np.ones((self.G, self.G), np.uint8)
        q = np.clip((np.asarray(Q, float) / self.cell).astype(int), 0, self.G - 1)
        empty[q[:, 1], q[:, 0]] = 0
        dist = cv2.distanceTransform(empty, cv2.DIST_L2, 5).astype(np.float32) * self.cell
        self.dist_flat = torch.cat([torch.as_tensor(dist, device=dev).reshape(-1), torch.full((1,), 1e6, device=dev)])
        # every ordered pair of points as a segment (start, length, angle)
        first, second = (torch.as_tensor(i, device=dev) for i in np.nonzero(~np.eye(len(Q), dtype=bool)))
        Qt = torch.as_tensor(Q, device=dev, dtype=torch.float32)
        seg = Qt[second] - Qt[first]
        self.seg_start, self.seg_len, self.seg_ang = Qt[first], torch.hypot(seg[:, 0], seg[:, 1]), torch.atan2(seg[:, 1], seg[:, 0])

    # ---- exact scoring -------------------------------------------------------------------
    def likelihood(self, pattern, nodes_in, pts_in, dists, P, tol, dof):
        """Log-likelihood ratio 'figure here' vs 'random stars'."""
        cfg, Q = self.cfg, self.Q
        n, m = len(pattern.nodes), len(Q)
        if m == 0:
            return -1e9
        lo, hi = Q.min(0), Q.max(0)
        area = max((hi[0] - lo[0] + 2 * tol) * (hi[1] - lo[1] + 2 * tol), 4e5)
        sigma = tol / cfg.id_sigma_div
        g = math.log(area / (m * 2 * math.pi * sigma * sigma))
        mg = cfg.id_off_margin
        far_out = int(((P[:, 0] < -mg) | (P[:, 0] > SKY_W + mg) | (P[:, 1] < -mg) | (P[:, 1] > SKY_H + mg)).sum())
        in_img = (P[:, 0] >= 5) & (P[:, 0] < SKY_W - 5) & (P[:, 1] >= 5) & (P[:, 1] < SKY_H - 5)
        score = float((g - dists ** 2 / (2 * sigma * sigma)).sum()) - dof * g     # inliers minus free parameters
        score -= cfg.id_off_penalty * g * far_out                                    # far outside the sky
        score -= cfg.id_offimg_w * g * float((~in_img).sum())                        # off-image nodes
        unmatched = in_img.copy()
        unmatched[nodes_in] = False
        if unmatched.any() and cfg.id_missing_w > 0:                                  # in-image nodes with no bright star
            d, _ = self.bright_tree.query(P[unmatched])
            score -= cfg.id_missing_w * g * float(np.clip((d - tol) / (2.0 * tol), 0.0, 1.0).sum())
        sizes = pattern.sizes
        if len(nodes_in) >= 4 and sizes.std() > 0.15 * (sizes.mean() + 1e-9) and self.qmag is not None and cfg.id_size_w > 0:
            rho = np.corrcoef(rankdata(sizes[nodes_in]), rankdata(self.qmag[pts_in]))[0, 1]
            if np.isfinite(rho):
                score += cfg.id_size_w * g * rho                                    # dot size ~ star brightness
        return float(score)

    def refine(self, pattern, A, t, flip) -> FigureFit:
        """Refit a seed transform on its inliers (affine if enough and well-behaved, else
        similarity) for as long as the fit improves, then score it exactly."""
        cfg, Q = self.cfg, self.Q
        s0 = math.sqrt(abs(np.linalg.det(A)))
        tol = min(cfg.id_tol_hi, max(cfg.id_tol_lo, cfg.id_tol_rel * s0))

        def acceptable(candidate):
            sv = np.linalg.svd(candidate[1], compute_uv=False)
            s2 = math.sqrt(sv[0] * sv[1])
            return sv[1] >= 1e-9 and sv[0] / sv[1] <= cfg.id_max_aniso and 0.7 * s0 <= s2 <= 1.4 * s0

        current = match_nodes(A, t, pattern, Q, self.tree, cfg.id_seed_mult * tol)
        dof, accepted = 2, False
        for _ in range(6):
            nodes_in, pts_in, dd, _ = current
            if len(nodes_in) < 3:
                break
            src, dst = pattern.nodes[nodes_in], Q[pts_in]
            candidates = ([(3,) + fit_affine(src, dst)] if len(nodes_in) >= cfg.id_affine_min else [])
            candidates.append((2,) + fit_similarity(src, dst, flip))
            chosen = next(filter(acceptable, candidates), None)
            if chosen is None:
                break
            trial = match_nodes(chosen[1], chosen[2], pattern, Q, self.tree, tol)
            more = len(trial[0]) > len(nodes_in)
            tighter = len(trial[0]) == len(nodes_in) and (trial[2] ** 2).sum() < (dd ** 2).sum() - 1e-6
            if accepted and not (more or tighter):
                break
            dof, A, t = chosen
            current, accepted = trial, True
        if not accepted:
            current = match_nodes(A, t, pattern, Q, self.tree, tol)
        nodes_in, pts_in, dd, P = current
        if dof == 3:     # charge an affine fit in proportion to how anisotropic it is
            sv = np.linalg.svd(A, compute_uv=False)
            dof = 2.0 + min(1.0, math.log(max(1.0, sv[0] / max(sv[1], 1e-9))) / math.log(cfg.id_max_aniso))
        return FigureFit(name=pattern.name, score=self.likelihood(pattern, nodes_in, pts_in, dd, P, tol, dof),
                         n_in=len(nodes_in), n_nodes=len(pattern.nodes), A=A, t=t, in_nodes=nodes_in, in_pts=pts_in,
                         tol=tol, resid=float(dd.mean()) if len(dd) else 0.0)

    # ---- bulk screening ------------------------------------------------------------------
    def _screen(self, pattern):
        """Score every (seed pair, ordered point pair, handedness) on the device.  Returns the
        per-chunk shortlists [soft, count, flip, scale, angle, tx, ty] and the inlier histogram."""
        cfg, dev = self.cfg, self.dev
        n = len(pattern.nodes)
        G, cell, outside = self.G, self.cell, self.G * self.G
        margin, off_m = cfg.id_shift_margin, cfg.id_off_margin
        max_off = max(cfg.id_max_off, int(cfg.id_max_off_frac * n))
        pairs = torch.as_tensor(pattern.pairs, device=dev)
        histogram = torch.zeros(n + 1, dtype=torch.int64, device=dev)
        per_chunk = max(1, int(cfg.id_chunk_work // max(1, len(self.seg_len) * n)))
        shortlist = []
        for flip in (1.0, -1.0):
            nodes = torch.as_tensor(pattern.nodes * np.array([1.0, flip]), device=dev, dtype=torch.float32)
            nx, ny = nodes[None, :, 0], nodes[None, :, 1]
            for c0 in range(0, len(pairs), per_chunk):
                pr = pairs[c0:c0 + per_chunk]
                na, nb = nodes[pr[:, 0]], nodes[pr[:, 1]]
                dn = nb - na
                s = self.seg_len[None, :] / torch.hypot(dn[:, 0], dn[:, 1])[:, None]
                th = self.seg_ang[None, :] - torch.atan2(dn[:, 1], dn[:, 0])[:, None]
                c, si = torch.cos(th), torch.sin(th)
                tx = self.seg_start[None, :, 0] - s * (c * na[:, None, 0] - si * na[:, None, 1])
                ty = self.seg_start[None, :, 1] - s * (si * na[:, None, 0] + c * na[:, None, 1])
                ok = (s >= cfg.id_scale_lo) & (s <= cfg.id_scale_hi)
                ok &= (tx > -margin) & (tx < SKY_W + margin) & (ty > -margin) & (ty < SKY_H + margin)
                if not bool(ok.any()):
                    continue
                s, th, tx, ty, c, si = s[ok], th[ok], tx[ok], ty[ok], c[ok], si[ok]
                X = tx[:, None] + s[:, None] * (c[:, None] * nx - si[:, None] * ny)
                Y = ty[:, None] + s[:, None] * (si[:, None] * nx + c[:, None] * ny)
                in_img = (X >= 0) & (X < SKY_W) & (Y >= 0) & (Y < SKY_H)
                n_off = ((X < -off_m) | (X > SKY_W + off_m) | (Y < -off_m) | (Y > SKY_H + off_m)).sum(1)
                cells = (Y / cell).long().clamp(0, G - 1) * G + (X / cell).long().clamp(0, G - 1)
                d = self.dist_flat[torch.where(in_img, cells, torch.full_like(cells, outside))]
                tol = torch.clamp(cfg.id_tol_rel * s, min=cfg.id_tol_lo, max=cfg.id_tol_hi)
                count = (d <= (cfg.id_seed_mult * tol)[:, None]).sum(1)
                soft = torch.exp(-0.5 * (d / (cfg.id_soft_mult * tol)[:, None]) ** 2).sum(1)
                usable = (n_off <= max_off) & (count >= 3)
                count = torch.where(usable, count, torch.zeros_like(count))
                soft = torch.where(usable, soft, torch.zeros_like(soft))
                histogram += torch.bincount(count, minlength=n + 1)[: n + 1]
                k = min(cfg.id_keep_per_chunk, len(soft))
                top = torch.topk(soft, k).indices
                shortlist.append(torch.stack([soft[top], count[top].float(), torch.full((k,), flip, device=dev),
                                              s[top], th[top], tx[top], ty[top]], 1))
        return shortlist, histogram.cpu().numpy()

    @staticmethod
    def _near_duplicate(seen, key, th0):
        dth = np.abs((seen[:, 1] - th0 + math.pi) % (2 * math.pi) - math.pi) * 10
        return np.maximum(np.abs(seen[:, [0, 2, 3]] - key[[0, 2, 3]]).max(1), dth).min() < 1.0

    def fit(self, pattern) -> FigureFit:
        """Best transform of one figure: screen everything, then refine the top distinct poses."""
        cfg = self.cfg
        best = FigureFit(name=pattern.name, n_nodes=len(pattern.nodes))
        n = len(pattern.nodes)
        if n < 3 or len(self.Q) < 3 or len(pattern.pairs) == 0:
            return best
        shortlist, histogram = self._screen(pattern)
        occupied = np.nonzero(histogram[3:])[0]
        if not shortlist or len(occupied) == 0:
            return best
        ties = int(histogram[3 + occupied[-1]])          # hypotheses sharing the best inlier count
        budget = cfg.id_verify_min if n <= 4 else max(cfg.id_verify_min, min(cfg.id_verify_max, ties))
        rows = torch.cat(shortlist)
        rows = rows[torch.argsort(-rows[:, 0])][: max(cfg.id_keep_per_chunk, 4 * budget)].cpu().numpy()
        seen, checked = np.zeros((0, 4)), 0
        for _, count0, flip, s0, th0, tx0, ty0 in rows:
            if count0 < 3:
                continue
            th0 = (th0 + math.pi) % (2 * math.pi) - math.pi
            key = np.array([np.log(s0) * 20, th0, tx0 / 80.0, ty0 / 80.0])
            if len(seen) and self._near_duplicate(seen, key, th0):
                continue
            seen = np.vstack([seen, key])
            if checked >= budget:
                break
            checked += 1
            c0, si0 = math.cos(th0), math.sin(th0)
            A = s0 * np.array([[c0, -si0], [si0, c0]]) @ np.diag([1.0, flip])
            candidate = self.refine(pattern, A, np.array([tx0, ty0]), flip)
            if candidate.score > best.score:
                best = candidate
        best.n_checked = checked
        return best

    def rank(self, patterns):
        """(best fit, all fits sorted by calibrated score)."""
        cfg = self.cfg
        fits, null_scores = [], []
        for i, pattern in enumerate(patterns):
            fit = self.fit(pattern)
            fit.raw = max(float(fit.score), 0.0)
            rng = np.random.default_rng(cfg.null_seed + 1000 * i)
            null_scores.append([max(float(self.fit(pattern.scrambled(rng, cfg)).score), 0.0) for _ in range(cfg.null_k)])
            fits.append(fit)
        if cfg.null_k > 0:
            calibrate(fits, null_scores, patterns, cfg)
        fits.sort(key=lambda f: -f.score)
        return fits[0], fits


def identify_constellation(Q, qmag, patterns, cfg, bright_xy=None):
    if not patterns:
        return FigureFit(), []
    if len(Q) < 3:
        biggest = max(patterns, key=lambda p: len(p.nodes))
        return FigureFit(name=biggest.name, score=0.0, n_nodes=len(biggest.nodes)), []
    return Identifier(Q, qmag, cfg, DEVICE, bright_xy).rank(patterns)


def identification_points(stars, results, cfg):
    """Brightest catalogue stars plus the positions of present patches, merged within
    id_merge_px.  Brightness = saturated-pixel count, with flux as a tie-break.  Returns
    (Q, qflux, qsat, qmag, keep mask, brightest id_n_complete stars)."""
    by_flux = np.argsort(-stars.flux)
    top = by_flux[: cfg.id_n_stars]
    present = [r for r in results if r.present]
    extra_xy = [np.array([[r.x, r.y] for r in present])] if present else []
    extra_flux = [np.array([stars.flux[r.star] for r in present], float)] if present else []
    extra_sat = [np.array([stars.sat[r.star] for r in present], float)] if present else []
    Q = np.vstack([stars.xy[top].astype(float)] + extra_xy)
    qflux = np.concatenate([stars.flux[top].astype(float)] + extra_flux)
    qsat = np.concatenate([stars.sat[top].astype(float)] + extra_sat)
    keep = greedy_suppress(Q, cfg.id_merge_px, qflux)
    qmag = qsat + (qflux / (qflux.max() + 1.0) if len(qflux) else qflux)
    return Q, qflux, qsat, qmag, keep, stars.xy[by_flux[: cfg.id_n_complete]].astype(float)


def loose_refit(pattern, fit, Q, cfg):
    """Refit with a looser tolerance, used only to place nodes on stars (scoring stays tight)."""
    if fit.A is None or len(Q) < 3:
        return fit.A, fit.t
    flip = 1.0 if np.linalg.det(fit.A) >= 0 else -1.0
    nodes_in, pts_in, _, _ = match_nodes(fit.A, fit.t, pattern, Q, cKDTree(Q), max(fit.tol, cfg.id_tol_lo) * cfg.place_tol_mult)
    if len(nodes_in) < max(3, fit.n_in):
        return fit.A, fit.t
    return fit_similarity(pattern.nodes[nodes_in], Q[pts_in], flip)

## 10. Putting leftover bright patches on the figure

With the constellation named, its transform is refitted with a 3× looser tolerance (for placement only) and each projected node snaps to the brightest catalogue star within 30 px.

A bright patch that leads its runner-up by at least 0.08 is trusted as is. Every other bright patch is handled in three passes:

1. if it already lies within 12 px of a free node, it keeps its position and claims that node;
2. the remaining ones are assigned greedily, one per node, by their correlation at each node star;
3. fully blown-out patches (mean ≥ 150) cannot be correlated at all, so they take the brightest free nodes in brightness order.

Putting a point on a figure star can only increase geometric recovery. Finally, every present point within 12 px of a node is labelled `m = 1`.

In [ ]:
def snap_nodes_to_stars(P, sky, cfg):
    """(star positions, node indices, star fluxes): each in-image projected node taken to the
    brightest catalogue star within node_snap_px."""
    hits = []
    for i, (px, py) in enumerate(P):
        if 5 <= px < SKY_W - 5 and 5 <= py < SKY_H - 5:
            near = sky.tree.query_ball_point((px, py), cfg.node_snap_px)
            if near:
                hits.append((i, max(near, key=lambda s: sky.flux[s])))
    xy = np.array([sky.xy[j] for _, j in hits], float).reshape(-1, 2)
    return xy, np.array([i for i, _ in hits], int), np.array([sky.flux[j] for _, j in hits], float)


def node_scores(patches, chosen, st, node_xy, cfg):
    """Ring-whitened correlation of each chosen patch at each node star."""
    n_nodes = len(node_xy)
    if n_nodes == 0 or not chosen:
        return np.zeros((len(chosen), n_nodes))
    dev = st.dev
    nxy = torch.as_tensor(np.asarray(node_xy, np.float32), device=dev)
    d2 = ((nxy[:, None, :] - st.xy[None, :, :]) ** 2).sum(-1)
    nearest = d2.argmin(1)
    sat = ((st.sat_area[nearest] > 0) & (d2[torch.arange(n_nodes, device=dev), nearest] <= 25.0)).cpu().numpy()
    rows = []
    for k in chosen:
        patch = patches[k]
        tile = torch.as_tensor(patch.img_c, device=dev)
        scorer = PoseScorer(st, tile, cfg, inner=0.0, whiten=cfg.whiten, bg_inner=patch.halo_r)
        rows.append(scan_star_list(scorer, nxy, sat, core_centroid(patch.raw, cfg), cfg, refine_n=n_nodes).v.cpu().numpy())
    return np.array(rows, float)


def undecided(r, cfg):
    if r.via_bright:
        return not (r.present and r.pool_margin >= cfg.bright_sure_margin)
    return not (r.present and (r.ncc >= cfg.pres_ncc_alone or (r.ncc - r.ncc_alt) >= cfg.dense_confident_margin))


def assign_to_nodes(results, patches, st, node_xy, node_flux, tiles, cfg):
    """Give undecided bright patches the named figure's node stars, one each.  Returns
    (patch, node, ncc) for every placement."""
    if len(node_xy) == 0:
        return []
    blown = [k for k, r in enumerate(results) if tiles[k].mean() >= cfg.blown_mean and undecided(r, cfg)]
    bright = [k for k, r in enumerate(results) if patches[k].halo_r >= cfg.bright_radius and undecided(r, cfg) and k not in blown]
    taken_nodes, placed = set(), []

    def place(k, j, ncc, support, move=True):
        r = results[k]
        if move:
            r.x, r.y = float(node_xy[j, 0]), float(node_xy[j, 1])
        r.present, r.m, r.support = True, 1, support
        taken_nodes.add(j)
        placed.append((k, int(j) if move else j, ncc))

    # pass 1: a patch already sitting on a free node keeps it
    if cfg.node_keep_px > 0:
        for k in sorted(bright + blown, key=lambda k: -results[k].ncc):
            r = results[k]
            if r.x < 0:
                continue
            d = np.hypot(node_xy[:, 0] - r.x, node_xy[:, 1] - r.y)
            j = int(np.argmin(d))
            if d[j] <= cfg.node_keep_px and j not in taken_nodes:
                place(k, j, float(r.ncc), -3, move=False)
                (bright if k in bright else blown).remove(k)

    # pass 2: greedy one-to-one assignment by correlation at the node stars
    if bright:
        table = node_scores(patches, bright, st, node_xy, cfg)
        ranked = np.sort(table, axis=1)
        margin = ranked[:, -1] - ranked[:, -2] if table.shape[1] > 1 else ranked[:, -1]
        served = set()
        for flat in np.argsort(-table, axis=None):
            i, j = np.unravel_index(flat, table.shape)
            if table[i, j] < cfg.node_min_ncc:
                break
            if i in served or j in taken_nodes or margin[i] < cfg.node_min_margin:
                continue
            if len(taken_nodes) >= len(node_xy):
                break
            served.add(i)
            results[bright[i]].ncc = float(table[i, j])
            place(bright[i], j, float(table[i, j]), -1)

    # pass 3: blown-out patches to the brightest free nodes, by brightness rank
    if blown and cfg.assign_blown:
        free = deque(j for j in np.argsort(-node_flux) if j not in taken_nodes)
        for k in sorted(blown, key=lambda k: -tiles[k].mean()):
            if not free:
                break
            j = free.popleft()
            if node_flux[j] < cfg.blown_flux_frac * node_flux.max():
                break
            place(k, j, -1.0, -2)
    return placed

## 11. Solving a scene end to end

`prepare_scene` does the CPU-heavy front-end (sky catalogue and patch descriptors). `solve_scene` then runs localisation, presence, naming and placement and returns a `SceneOutcome`. `process_scenes` keeps up to `prefetch` scenes being prepared on worker threads while the current one is solved, and turns a scene that crashes into an all-absent / `unknown` answer instead of stopping the run.

In [ ]:
@dataclass
class Scene:
    sky: SkyModel
    patches: list
    tiles: list
    R: float


@dataclass
class SceneOutcome:
    sid: str
    results: list                 # TileResult per patch, after placement
    fit: FigureFit                # the chosen constellation
    ranked: list = dc_field(default_factory=list)
    before: list = None           # TileResults before naming and placement
    Q: np.ndarray = None          # naming points (merged)
    qflux: np.ndarray = None
    qsat: np.ndarray = None
    node_xy: np.ndarray = None
    node_flux: np.ndarray = None
    stars: SimpleNamespace = None  # xy / flux / sat of the catalogue
    noise: float = 0.0

    @property
    def ranking(self):
        return [(h.name, h.score, h.raw, h.null_mean, h.null_sd, h.n_in, h.n_nodes) for h in self.ranked]


def load_scene(folder):
    images = glob.glob(os.path.join(folder, "*_image.png")) or glob.glob(os.path.join(folder, "*.png"))
    patch_files = sorted(glob.glob(os.path.join(folder, "patches", "patch_*.png")),
                         key=lambda f: int(re.findall(r"(\d+)", os.path.basename(f))[-1]))
    return cv2.imread(images[0], cv2.IMREAD_GRAYSCALE), [cv2.imread(f, cv2.IMREAD_GRAYSCALE) for f in patch_files]


def prepare_scene(sky_img, tiles, cfg) -> Scene:
    sky = SkyModel(sky_img, cfg)
    patches = [Patch(t, cfg) for t in tiles]
    reach = max([p.nb_r.max() for p in patches if len(p.nb_r)] + [10.0])
    R = float(reach * cfg.zoom_hi + 2.5)
    sky.index_neighbours(cfg, R)
    return Scene(sky, patches, tiles, R)


def solve_scene(scene: Scene, patterns, cfg, tag="") -> SceneOutcome:
    t0 = time.time()
    note = lambda msg: log(cfg, f"[{tag}] {msg}  ({time.time() - t0:.1f}s)")
    sky = scene.sky
    st = SkyTensors(sky, DEVICE)
    note(f"sky stars={sky.N} noise={sky.noise:.2f}")

    # 1. locate every patch and decide presence
    results = Localizer(scene.patches, st, cfg).run()
    for r in results:
        r.present, r.m = decide_presence(r, cfg), 0
    before = [copy.copy(r) for r in results]
    note(f"located {len(results)} tiles, present={sum(r.present for r in results)}")

    # 2. name the constellation
    Q, qflux, qsat, qmag, keep, bright = identification_points(sky, results, cfg)
    Q, qflux, qsat = Q[keep], qflux[keep], qsat[keep]
    fit, ranked = identify_constellation(Q, qmag[keep], patterns, cfg, bright_xy=bright) if cfg.do_identify else (FigureFit(), [])
    runner = f"  runner-up {ranked[1].name} {ranked[1].score:.2f}" if len(ranked) > 1 else ""
    note(f"id={fit.name} cal={fit.score:.2f} raw={fit.raw:.2f} null={fit.null_mean:.1f}+{fit.null_sd:.1f} "
         f"inl={fit.n_in}/{fit.n_nodes}{runner}  |Q|={int(keep.sum())}")

    # 3. place undecided bright patches on the figure and label members
    node_xy, node_flux = np.zeros((0, 2)), np.zeros(0)
    if fit.A is not None:
        pattern = next(p for p in patterns if p.name == fit.name)
        A, t = loose_refit(pattern, fit, Q, cfg)
        node_xy, _, node_flux = snap_nodes_to_stars(apply_transform(A, t, pattern.nodes), sky, cfg)
        placed = assign_to_nodes(results, scene.patches, st, node_xy, node_flux, scene.tiles, cfg)
        note(f"{len(node_xy)} node stars, {len(placed)} bright tiles placed")
        if len(node_xy):
            for r in results:
                if r.present and np.hypot(node_xy[:, 0] - r.x, node_xy[:, 1] - r.y).min() <= cfg.member_px:
                    r.m = 1
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return SceneOutcome(tag, results, fit, ranked, before, Q, qflux, qsat, node_xy, node_flux, sky.star_list(), sky.noise)


def process_scenes(jobs, patterns, cfg):
    """jobs: (scene id, folder) pairs.  Returns one SceneOutcome per job, in order."""
    outcomes, upcoming = [], iter(jobs)
    prepare = lambda folder: prepare_scene(*load_scene(folder), cfg)
    with ThreadPoolExecutor(max_workers=max(1, cfg.prefetch)) as pool:
        inflight = deque((sid, pool.submit(prepare, folder)) for sid, folder in itertools.islice(upcoming, cfg.prefetch))
        while inflight:
            sid, future = inflight.popleft()
            try:
                scene = future.result()
            except Exception:
                scene = None
                print(f"[{sid}] preparation FAILED: {traceback.format_exc()}", file=sys.stderr, flush=True)
            inflight.extend((nsid, pool.submit(prepare, folder)) for nsid, folder in itertools.islice(upcoming, 1))
            outcome = SceneOutcome(sid, [], FigureFit())
            if scene is not None:
                try:
                    outcome = solve_scene(scene, patterns, cfg, tag=sid)
                except Exception:
                    print(f"[{sid}] scene FAILED, writing all-absent/unknown: {traceback.format_exc()}", file=sys.stderr, flush=True)
            outcomes.append(outcome)
    return outcomes

## 12. Scoring and the submission file

`competition_score` re-implements the metric. `write_submission` fills the template in `sample_submission.csv`. `report_training` and `predict_validation` are the two entry points used below.

In [ ]:
def distance_credit(d):
    """Full credit within 12 px, none beyond 36 px, linear in between."""
    return float(np.clip((36.0 - d) / 24.0, 0.0, 1.0))


def f1_score(tp, fp, fn):
    denominator = 2 * tp + fp + fn
    if tp + fp + fn == 0:
        return 1.0
    return 2 * tp / denominator if denominator else 0.0


def competition_score(truth, pred, true_name, pred_name):
    """truth / pred: lists holding None or (x, y, m)."""
    pairs = list(zip(truth, pred))
    tp = sum(1 for t, q in pairs if t is not None and q is not None)
    fp = sum(1 for t, q in pairs if t is None and q is not None)
    fn = sum(1 for t, q in pairs if t is not None and q is None)
    tn = len(truth) - tp - fp - fn
    presence = 0.5 * (f1_score(tp, fp, fn) + f1_score(tn, fn, fp))

    labelled = [(t, q) for t, q in pairs if t is not None]
    localisation = sum(distance_credit(math.hypot(t[0] - q[0], t[1] - q[1])) for t, q in labelled if q is not None)
    localisation = localisation / len(labelled) if labelled else 1.0

    figure = [t for t in truth if t is not None and t[2] == 1]
    reported = [q for q in pred if q is not None]
    geometric = 0.0
    if figure and reported:                 # greedy one-to-one matching, closest pairs first
        D = np.array([[math.hypot(a[0] - b[0], a[1] - b[1]) for b in reported] for a in figure])
        used_f, used_r = set(), set()
        for i, j in zip(*np.unravel_index(np.argsort(D, axis=None), D.shape)):
            if i not in used_f and j not in used_r:
                used_f.add(i)
                used_r.add(j)
                geometric += distance_credit(D[i, j])
    geometric = geometric / len(figure) if figure else 1.0

    identification = 1.0 if pred_name == true_name else 0.0
    return dict(presence=presence, localization=localisation, geometric=geometric, identification=identification,
                total=0.25 * presence + 0.20 * localisation + 0.25 * geometric + 0.30 * identification)


def parse_entry(text):
    """'(x, y, m)' -> (x, y, m); '-1', blanks and negative coordinates -> None."""
    numbers = [float(v) for v in re.findall(r"-?\d+(?:\.\d+)?", (text or "").strip())]
    if len(numbers) < 2 or numbers[0] < 0 or numbers[1] < 0:
        return None
    return (numbers[0], numbers[1], int(numbers[2]) if len(numbers) >= 3 else 0)


def load_truth(path):
    with open(path, newline="") as fh:
        return {row["Id"]: ([parse_entry(row[f"patch_{i:02d}"]) for i in range(1, int(row["n_patches"]) + 1)],
                            row["constellation"].strip())
                for row in csv.DictReader(fh)}


def as_cells(results):
    """Integer, image-clamped cells exactly as they are written to the CSV."""
    clamp = lambda v, hi: int(min(max(round(v), 0), hi - 1))
    return [(clamp(r.x, SKY_W), clamp(r.y, SKY_H), int(r.m)) if r.present else None for r in results]


def format_entry(cell):
    return "-1" if cell is None else f"({int(round(cell[0]))}, {int(round(cell[1]))}, {int(cell[2])})"


def search_presence_thresholds(outcomes, truth, cfg):
    """Optional grid search of the presence thresholds on train (0.25 P + 0.20 L + 0.25 G).
    Prints the best setting; not used by default."""
    grid = dict(pres_ncc_alone=[0.7, 0.75, 0.8, 0.99], pres_ncc_floor=[0.55, 0.6, 0.65, 0.7],
                pres_margin=[0.03, 0.06, 0.08, 0.12], pres_z_alone=[6.5, 8.0, 99.0],
                pres_bright_ncc=[0.45, 0.55, 0.65, 99.0])

    def partial_score(trial):
        parts = []
        for o in outcomes:
            cells, name = truth[o.sid]
            pred = [(r.x, r.y, r.m) if decide_presence(r, trial) else None for r in o.results]
            s = competition_score(cells, pred, name, "unknown")
            parts.append(0.25 * s["presence"] + 0.20 * s["localization"] + 0.25 * s["geometric"])
        return np.mean(parts)

    best, best_v = None, -1.0
    for values in itertools.product(*grid.values()):
        setting = dict(zip(grid, values))
        v = partial_score(configure(cfg, **setting))
        if v > best_v:
            best_v, best = float(v), setting
    print(f"\n[tune] best presence thresholds (partial score {best_v:.3f}): " + " ".join(f"{k}={v}" for k, v in best.items()))
    return best


def write_submission(outcomes, data_dir, out_path):
    with open(os.path.join(data_dir, "sample_submission.csv"), newline="") as fh:
        header, *template = list(csv.reader(fh))
    answers = {o.sid: (as_cells(o.results), o.fit.name or "unknown") for o in outcomes}
    slots = len(header) - 3
    with open(out_path, "w", newline="") as fh:
        writer = csv.writer(fh, quoting=csv.QUOTE_MINIMAL)
        writer.writerow(header)
        for sid, n_patches, *_ in template:
            cells, name = answers.get(sid, ([], "unknown"))
            padded = cells[:slots] + [None] * (slots - len(cells[:slots]))
            writer.writerow([sid, str(int(n_patches))] + [format_entry(c) for c in padded] + [name])
    print("wrote", out_path)


def print_scene_report(o: SceneOutcome, cells, true_name, s, show_tiles):
    print(f"\n=== {o.sid}: true={true_name} pred={o.fit.name}  " + "  ".join(f"{k}={v:.3f}" for k, v in s.items()))
    if o.ranked:
        print("    top-5 (calibrated; raw, null mean+sd): " + ", ".join(
            f"{h.name}({h.score:.2f}; {h.raw:.1f}, {h.null_mean:.1f}+{h.null_sd:.1f}, {h.n_in}/{h.n_nodes})" for h in o.ranked[:5]))
    true_fit = next((h for h in o.ranked if h.name == true_name and h.A is not None), None)
    if true_fit is not None:
        sv = np.linalg.svd(true_fit.A, compute_uv=False)
        print(f"    true-figure fit: inliers={true_fit.n_in}/{true_fit.n_nodes} scale={math.sqrt(sv[0] * sv[1]):.0f}px/unit "
              f"anisotropy={sv[0] / max(sv[1], 1e-9):.2f} resid={true_fit.resid:.1f}px tol={true_fit.tol:.1f}px")
    zooms = [r.zoom for r, t in zip(o.results, cells) if t is not None and r.present]
    if zooms:
        print(f"    recovered zooms: min={min(zooms):.2f} med={np.median(zooms):.2f} max={max(zooms):.2f}")
    if not show_tiles:
        return
    print("    tile  truth  pres  n_nb sup   ncc   alt      z    bgc  z_bg  nbg  zoom     err")
    for k, (r, t) in enumerate(zip(o.results, cells)):
        err = math.hypot(t[0] - r.x, t[1] - r.y) if (t is not None and r.x >= 0) else float("nan")
        kind = "abs" if t is None else ("fig" if t[2] == 1 else "off")
        print(f"    {k + 1:02d}     {kind}    {int(r.present)}    {r.n_nb}   {r.support}  {r.ncc:5.2f}  {r.ncc_alt:5.2f}  "
              f"{r.z:5.1f}  {r.bgc:5.2f} {r.z_bgc:5.1f}  {r.n_bgc:3.0f}  {r.zoom:5.2f}  {err:7.1f}")


def report_training(data_dir, patterns, cfg, show_tiles=True, tune=False, limit=0):
    """Solve the labelled scenes; print the metric, the top-5 names and a per-patch table."""
    truth = load_truth(os.path.join(data_dir, "train_ground_truth.csv"))
    jobs = [(sid, os.path.join(data_dir, "train", sid)) for sid in truth][: (limit or None)]
    outcomes = process_scenes(jobs, patterns, cfg)
    scores = []
    for o in outcomes:
        cells, true_name = truth[o.sid]
        scores.append(competition_score(cells, as_cells(o.results), true_name, o.fit.name))
        print_scene_report(o, cells, true_name, scores[-1], show_tiles)
    print("\nMEAN: " + "  ".join(f"{k}={np.mean([s[k] for s in scores]):.3f}" for k in scores[0]))
    return outcomes, (search_presence_thresholds(outcomes, truth, cfg) if tune else None)


def predict_validation(data_dir, patterns, cfg, out_path="submission.csv", limit=0):
    with open(os.path.join(data_dir, "sample_submission.csv"), newline="") as fh:
        ids = [row[0] for row in list(csv.reader(fh))[1:]]
    jobs = [(sid, os.path.join(data_dir, "validation", sid)) for sid in ids][: (limit or None)]
    outcomes = process_scenes(jobs, patterns, cfg)
    write_submission(outcomes, data_dir, out_path)
    return outcomes

## 13. Load the reference figures

On a GPU the sweep runs at full resolution; on a CPU it uses stride 2.

In [ ]:
settings = configure(search_stride=1 if DEVICE.type == "cuda" else 2, mem_budget_gb=4.0, search_batch=6, corr_batch=16384)
pattern_bank = load_patterns(DATA, settings) if DATA else []
print("device:", DEVICE, "| search stride:", settings.search_stride, "| figures:", len(pattern_bank))

## 14. Diagnostics on the Pisces training scene

These cells only draw pictures and are skipped when the file runs as a script. The helper below loads and preprocesses a training scene once and caches it.

In [ ]:
if IN_NOTEBOOK:
    import matplotlib.pyplot as plt

    _SCENE_CACHE = globals().setdefault("_SCENE_CACHE", {})

    def demo_scene(name="pisces"):
        """(sky image, patches, prepared Scene) of a training scene, cached."""
        if name not in _SCENE_CACHE:
            sky_img, tiles = load_scene(os.path.join(DATA, "train", name))
            _SCENE_CACHE[name] = (sky_img, tiles, prepare_scene(sky_img, tiles, settings))
        return _SCENE_CACHE[name]

### A. Where the pixel values sit, stage by stage

Histograms (top) and cumulative distributions (bottom) of the raw sky, the background-subtracted sky, and the square-root-stretched image that the correlation actually sees. The raw values pile up at the dark end; background removal centres them on zero; the stretch pulls in the bright tail without changing the order of intensities. Histogram equalisation would have destroyed the brightness ratios, which is why it is not used.

In [ ]:
if IN_NOTEBOOK:
    _t = time.time()
    sky_img, tiles, scene = demo_scene()
    hp = subtract_background(sky_img, settings)
    stages = [("raw sky", sky_img[::3, ::3].ravel().astype(np.float32), (0, 255), 256),
              ("minus median background", hp[::3, ::3].ravel(), (-40, 200), 240),
              ("sqrt stretch (noise units)", scene.sky.corr_image[::3, ::3].ravel(), (-3, 8), 220)]
    fig, ax = plt.subplots(2, 3, figsize=(13, 6))
    for col, (title, values, span, bins) in enumerate(stages):
        counts, edges = np.histogram(values, bins=bins, range=span)
        centres = 0.5 * (edges[1:] + edges[:-1])
        top, bottom = ax[0, col], ax[1, col]
        top.semilogy(centres, np.maximum(counts, 1), lw=1.2)
        top.set_title(title, fontsize=10)
        bottom.plot(centres, counts.cumsum() / max(counts.sum(), 1), lw=1.5)
        for q in (0.5, 0.99, 0.999):
            xq = float(np.percentile(values, 100 * q))
            bottom.axvline(xq, color="k", ls=":", lw=0.8)
            bottom.text(xq, q - 0.08, f"{q:g}: {xq:.1f}", fontsize=7, rotation=90, va="top")
        for a in (top, bottom):
            a.set_xlim(span)
            a.grid(True, alpha=0.3)
        bottom.set_ylim(0, 1.02)
    ax[0, 0].set_ylabel("pixels (log)")
    ax[1, 0].set_ylabel("CDF")
    inset = ax[0, 2].inset_axes([0.55, 0.5, 0.42, 0.45])
    xx = np.linspace(-5, 60, 200)
    inset.plot(xx, np.sign(xx) * np.sqrt(np.abs(xx)), color="C3", lw=1.2)
    inset.set_title("lookup: sign(x)·sqrt|x|", fontsize=7)
    inset.tick_params(labelsize=6)
    fig.suptitle(f"pisces: {(sky_img >= 250).mean() * 100:.3f}% saturated pixels, noise sigma {robust_sigma(hp[::7, ::7]):.1f}", fontsize=10)
    plt.tight_layout()
    plt.show()
    print(f"plot A: {time.time() - _t:.1f}s")

### B. Matched filter vs DoG vs LoG

One crop, three detectors at the same σ: the Gaussian matched filter used by the pipeline, a difference of Gaussians, and a scale-normalised Laplacian of Gaussian. Green rings are labelled present patches, cyan rings are detections. The printout measures how many of each detector's 350 brightest stars coincide with the matched filter's.

In [ ]:
if IN_NOTEBOOK:
    _t = time.time()
    sky_img, tiles, scene = demo_scene()
    truth_cells, _ = load_truth(os.path.join(DATA, "train_ground_truth.csv"))["pisces"]
    labelled = np.array([c[:2] for c in truth_cells if c is not None], float)
    members = np.array([c[:2] for c in truth_cells if c is not None and c[2] == 1], float)
    h, w = sky_img.shape
    hp = subtract_background(sky_img, settings)
    clean = cv2.medianBlur(hp, 3)
    sg = settings.star_psf_sigma
    g1 = cv2.GaussianBlur(clean, (0, 0), sg)
    responses = {"matched filter (pipeline)": g1,
                 "DoG  G(1.2) - G(1.9)": g1 - cv2.GaussianBlur(clean, (0, 0), 1.6 * sg),
                 "LoG  -sigma^2 Laplacian": -sg * sg * cv2.Laplacian(g1, cv2.CV_32F, ksize=3)}
    detections = {}
    for name, resp in responses.items():
        nz = robust_sigma(resp[::7, ::7])
        detections[name] = (detect_peaks(resp, clean, settings.star_k_sigma * nz, settings.star_nms_radius, settings.star_cap, border=3), nz)

    def in_window(x, y, x0, y0):
        return (x >= x0) & (x < x0 + 400) & (y >= y0) & (y < y0 + 400)

    # the 400 x 400 window, centred on a figure star, that holds the most labelled stars
    windows = [(int(np.clip(c[0] - 200, 0, w - 400)), int(np.clip(c[1] - 200, 0, h - 400))) for c in members]
    _, x0, y0 = max((int(in_window(labelled[:, 0], labelled[:, 1], x0, y0).sum()), x0, y0) for x0, y0 in windows)
    win = (slice(y0, y0 + 400), slice(x0, x0 + 400))
    fig, ax = plt.subplots(1, 4, figsize=(14, 4.2))
    ax[0].imshow(np.sign(hp[win]) * np.sqrt(np.abs(hp[win])), cmap="gray", vmin=-2, vmax=12)
    ax[0].set_title("background-free crop", fontsize=8)
    for a, (name, resp) in zip(ax[1:], responses.items()):
        stars, nz = detections[name]
        a.imshow(resp[win] / nz, cmap="magma", vmin=-2, vmax=15)
        inside = in_window(stars.x, stars.y, x0, y0)
        a.scatter(stars.x[inside] - x0, stars.y[inside] - y0, s=26, facecolors="none", edgecolors="cyan", lw=0.7)
        a.set_title(f"{name}\n{int(inside.sum())} in crop, {len(stars)} in sky", fontsize=8)
    for a in ax:
        a.scatter(labelled[:, 0] - x0, labelled[:, 1] - y0, s=120, facecolors="none", edgecolors="lime", lw=1.2)
        a.set_xlim(0, 400)
        a.set_ylim(400, 0)
        a.set_xticks([])
        a.set_yticks([])
    plt.tight_layout()
    plt.show()
    reference = detections["matched filter (pipeline)"][0]
    ref_tree = cKDTree(reference.xy)
    ref_top = np.argsort(-reference.flux)[:350]
    for name, (stars, _) in detections.items():
        d, j = ref_tree.query(stars.xy[np.argsort(-stars.flux)[:350]])
        shared = np.isin(j[d <= 2.0], ref_top).sum()
        worst = cKDTree(stars.xy).query(members)[0].max()
        print(f"{name:28s} N={len(stars):6d}  top-350 shared with matched filter {shared / 3.5:5.1f}%  worst figure-star distance {worst:.2f}px")
    print(f"plot B: {time.time() - _t:.1f}s")

### C. Why textbook `TM_CCOEFF_NORMED` is not enough for a bright patch

The standard recipe, `cv2.matchTemplate` plus `minMaxLoc` over a rotation sweep, is run on a bright figure patch. Its maximum lands on the wrong star, because every bright star has the same round halo. The two correlations this pipeline uses are then evaluated at the true location and at that false peak, and compared with their random-pose null. Ground truth is used here only for drawing.

In [ ]:
if IN_NOTEBOOK:
    _t = time.time()
    sky_img, tiles, scene = demo_scene()
    patches = scene.patches
    truth_cells, _ = load_truth(os.path.join(DATA, "train_ground_truth.csv"))["pisces"]

    def textbook_match(image, tile, crop=22):
        """(score, centre, angle, response map) of the best TM_CCOEFF_NORMED peak over 36 rotations."""
        results = []
        for a in range(0, 360, 10):
            rot = cv2.warpAffine(tile, cv2.getRotationMatrix2D((15.5, 15.5), a, 1.0), (32, 32),
                                 flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)[5:5 + crop, 5:5 + crop]
            res = cv2.matchTemplate(image, np.ascontiguousarray(rot), cv2.TM_CCOEFF_NORMED)
            _, mv, _, ml = cv2.minMaxLoc(res)
            results.append((mv, (ml[0] + crop // 2, ml[1] + crop // 2), a, res))
        return max(results, key=lambda r: r[0])

    bright_members = sorted((k for k, (p, t) in enumerate(zip(patches, truth_cells))
                             if t is not None and t[2] == 1 and p.halo_r >= settings.bright_radius), key=lambda k: -patches[k].halo_r)
    for k in bright_members[:3]:
        mv, loc, ang, res = textbook_match(scene.sky.corr_image, patches[k].img_c)
        err = math.hypot(loc[0] - truth_cells[k][0], loc[1] - truth_cells[k][1])
        if err > 36:
            break
    if err <= 36:
        print("note: the textbook recipe happened to find this tile")
    tx, ty = float(truth_cells[k][0]), float(truth_cells[k][1])
    at_truth = float(res[int(np.clip(round(ty) - 11, 0, res.shape[0] - 1)), int(np.clip(round(tx) - 11, 0, res.shape[1] - 1))])
    st = SkyTensors(scene.sky, DEVICE)
    tile = torch.as_tensor(patches[k].img_c, device=DEVICE)
    zooms = np.array(settings.node_zooms, np.float32)
    A = torch.as_tensor(np.repeat(np.arange(36) * 2 * np.pi / 36, len(zooms)), dtype=torch.float32, device=DEVICE)
    Z = torch.as_tensor(np.tile(zooms, 36), dtype=torch.float32, device=DEVICE)
    g = torch.Generator()
    g.manual_seed(0)
    n_null = min(settings.null_draws, st.N)
    pick = torch.randperm(st.N, generator=g)[:n_null]
    ra = torch.rand(n_null, generator=g) * 2 * math.pi
    rz = torch.exp(torch.rand(n_null, generator=g) * (math.log(1.25) - math.log(0.8)) + math.log(0.8))
    variants = {"ring-equalised full disc (bright path)": dict(whiten=True, inner=0.0),
                "halo-free annulus (dense path)": dict(whiten=False, inner=max(settings.corr_inner_r, patches[k].halo_r))}
    rows = {}
    for label, kw in variants.items():
        scorer = PoseScorer(st, tile, settings, bg_inner=patches[k].halo_r, **kw)
        best_at = lambda x, y: float(scorer.ncc(torch.full_like(A, float(x)), torch.full_like(A, float(y)), Z, A).max())
        rows[label] = (best_at(tx, ty), best_at(*loc), scorer.ncc(st.xy[pick, 0], st.xy[pick, 1], rz, ra).numpy())
    fig = plt.figure(figsize=(15, 4.8))
    gs = fig.add_gridspec(1, 4, width_ratios=[0.8, 1.6, 1.4, 1.4])
    a0 = fig.add_subplot(gs[0])
    a0.imshow(tiles[k], cmap="gray")
    a0.set_xticks([])
    a0.set_yticks([])
    a0.set_title(f"patch_{k + 1:02d}, halo {patches[k].halo_r:.0f}px", fontsize=9)
    a1 = fig.add_subplot(gs[1])
    a1.imshow(cv2.resize(res, (res.shape[1] // 4, res.shape[0] // 4), interpolation=cv2.INTER_AREA), cmap="viridis",
              extent=[11, 11 + res.shape[1], 11 + res.shape[0], 11], vmin=-0.2, vmax=1)
    a1.scatter([tx], [ty], s=160, facecolors="none", edgecolors="lime", lw=1.5, label=f"truth: {at_truth:.2f}")
    a1.scatter([loc[0]], [loc[1]], s=160, marker="s", facecolors="none", edgecolors="red", lw=1.5, label=f"peak: {mv:.2f} ({err:.0f}px off)")
    a1.legend(fontsize=7, loc="lower left")
    a1.set_title(f"TM_CCOEFF_NORMED, best of 36 angles ({ang} deg)", fontsize=9)
    for spec, (label, (v_t, v_f, null)) in zip((gs[2], gs[3]), rows.items()):
        a = fig.add_subplot(spec)
        a.hist(null, bins=30, color="0.75", label="200 random poses")
        a.axvline(v_t, color="lime", lw=2, label=f"truth {v_t:.2f}")
        a.axvline(v_f, color="red", lw=2, ls="--", label=f"false peak {v_f:.2f}")
        a.set_title(label, fontsize=9)
        a.legend(fontsize=7)
    plt.tight_layout()
    plt.show()
    for label, (v_t, v_f, null) in rows.items():
        med = np.median(null)
        mad = 1.4826 * np.median(np.abs(null - med)) + 1e-3
        print(f"{label:40s} truth {v_t:.3f}  false {v_f:.3f}  null max {null.max():.3f}  z(truth) {(v_t - med) / mad:.1f}")
    print(f"plot C: {time.time() - _t:.1f}s")

### D. The evidence behind each presence decision

Values are taken *before* naming and placement. Left: best vs runner-up correlation for ordinary patches. Middle: best correlation vs z-score for ordinary patches. Right: bright-star patches. The lines are the thresholds of the presence rules. If the training run in section 15 has already happened its results are reused; otherwise the Pisces patches are located on the spot with naming switched off.

In [ ]:
if IN_NOTEBOOK:
    _t = time.time()
    if globals().get("train_outcomes"):
        source, outcomes = "evaluate_train", train_outcomes
    else:
        print("no train results yet: locating pisces now (naming off)")
        sky_img, tiles, scene = demo_scene()
        source, outcomes = "live pisces run", [solve_scene(scene, [], configure(settings, do_identify=False, verbose=0), tag="pisces")]
    truth = load_truth(os.path.join(DATA, "train_ground_truth.csv"))
    rows = [dict(kind="absent" if t is None else ("figure" if t[2] == 1 else "off-figure"),
                 ncc=r.ncc, alt=r.ncc_alt, z=r.z, support=r.support,
                 bright=r.halo_r >= settings.bright_radius, said=decide_presence(r, settings), truth=t is not None)
            for o in outcomes for r, t in zip(o.before or o.results, truth[o.sid][0])]
    colour = {"figure": "tab:green", "off-figure": "tab:blue", "absent": "tab:red"}
    fig, ax = plt.subplots(1, 3, figsize=(16, 5))
    for kind, col in colour.items():
        style = dict(c=col, marker="x" if kind == "absent" else "o", s=28, alpha=0.85)
        plain = [r for r in rows if r["kind"] == kind and not r["bright"]]
        bright = [r for r in rows if r["kind"] == kind and r["bright"]]
        ax[0].scatter([r["ncc"] for r in plain], [r["alt"] for r in plain], label=f"{kind} ({len(plain)})", **style)
        ax[1].scatter([r["ncc"] for r in plain], [r["z"] for r in plain], **style)
        ax[2].scatter([r["ncc"] for r in bright], [r["alt"] for r in bright], label=f"{kind} ({len(bright)})", **style)
    xs = np.linspace(settings.pres_ncc_floor, settings.pres_ncc_alone, 20)
    ax[0].axvline(settings.pres_ncc_alone, color="k", lw=1, label=f"ncc >= {settings.pres_ncc_alone}")
    ax[0].plot(xs, xs - settings.pres_margin, "k--", lw=1, label=f"lead >= {settings.pres_margin}")
    ax[0].axvline(settings.pres_ncc_floor, color="k", ls=":", lw=0.8, label=f"ncc >= {settings.pres_ncc_floor}")
    ax[1].axhline(settings.pres_z_alone, color="k", lw=1, label=f"z >= {settings.pres_z_alone}")
    ax[1].axhline(settings.pres_z_weak, color="k", ls="--", lw=1, label=f"z >= {settings.pres_z_weak} with >= {settings.pres_min_support} votes")
    voted = [r for r in rows if not r["bright"] and r["support"] >= settings.pres_min_support]
    ax[1].scatter([r["ncc"] for r in voted], [r["z"] for r in voted], s=110, facecolors="none", edgecolors="k", lw=0.8, label="enough votes")
    for a, xl, yl, title in ((ax[0], "best ncc", "runner-up ncc (> 5 px away)", "ordinary tiles: lead test"),
                             (ax[1], "best ncc", "z vs random poses", "ordinary tiles: null test"),
                             (ax[2], "best candidate-star ncc", "runner-up star", "bright tiles: present by prior")):
        a.set_xlabel(xl)
        a.set_ylabel(yl)
        a.set_title(title, fontsize=10)
        a.legend(fontsize=7, loc="upper left")
        a.set_xlim(0, 1)
    for a in (ax[0], ax[2]):
        a.plot([0, 1], [0, 1], color="0.6", lw=0.6)
        a.set_ylim(0, 1)
    fig.suptitle(f"{len(rows)} training tiles before naming (source: {source})", fontsize=10)
    plt.tight_layout()
    plt.show()
    for flag in (False, True):
        group = [r for r in rows if r["bright"] == flag]
        tp = sum(r["said"] and r["truth"] for r in group)
        fp = sum(r["said"] and not r["truth"] for r in group)
        fn = sum(not r["said"] and r["truth"] for r in group)
        print(f"{'bright' if flag else 'ordinary'} tiles: {len(group):3d}  TP {tp:3d}  FP {fp:3d}  FN {fn:3d}  TN {len(group) - tp - fp - fn:3d}")
    print(f"plot D: {time.time() - _t:.1f}s")

### E. The winning figure drawn on the sky

The named figure is projected onto the Pisces sky. Matched nodes are joined to their stars and nodes with no star nearby are ringed in red. Alongside: the source drawing with the same nodes marked, and the calibrated top 5 (`raw − null mean − null sd`).

In [ ]:
if IN_NOTEBOOK:
    _t = time.time()
    outcome = next((o for o in (globals().get("train_outcomes") or []) if o.sid == "pisces" and o.fit.A is not None), None)
    if outcome is None:
        print("no train results yet: naming pisces from its bright stars now (a few minutes on CPU)")
        sky_img, tiles, scene = demo_scene()
        Q, qflux, qsat, qmag, keep, bright = identification_points(scene.sky, [], settings)
        fit, ranked = identify_constellation(Q[keep], qmag[keep], pattern_bank, settings, bright_xy=bright)
        outcome = SceneOutcome("pisces", [], fit, ranked, Q=Q[keep])
    fit = outcome.fit
    pattern = next(p for p in pattern_bank if p.name == fit.name)
    Q = np.asarray(outcome.Q, float)
    scale = math.sqrt(abs(np.linalg.det(fit.A)))
    tol = min(settings.id_tol_hi, max(settings.id_tol_lo, settings.id_tol_rel * scale))
    n_in, p_in, dd, P = match_nodes(fit.A, fit.t, pattern, Q, cKDTree(Q), tol)
    sky_img, _ = load_scene(os.path.join(DATA, "train", "pisces"))
    small = cv2.resize(sky_img, (sky_img.shape[1] // 4, sky_img.shape[0] // 4), interpolation=cv2.INTER_AREA)
    fig = plt.figure(figsize=(16, 6))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.5, 1.0, 1.1])

    a = fig.add_subplot(gs[0])
    a.imshow(np.sqrt(small / 255.0), cmap="gray", extent=[0, SKY_W, SKY_H, 0], vmin=0, vmax=1)
    a.scatter(Q[:, 0], Q[:, 1], s=4, c="0.7", label=f"{len(Q)} naming points")
    in_img = (P[:, 0] >= 0) & (P[:, 0] < SKY_W) & (P[:, 1] >= 0) & (P[:, 1] < SKY_H)
    lost = np.ones(len(P), bool)
    lost[n_in] = False
    a.scatter(P[n_in, 0], P[n_in, 1], s=90, facecolors="none", edgecolors="lime", lw=1.5, label=f"{len(n_in)}/{len(P)} nodes on a star")
    a.scatter(P[lost & in_img, 0], P[lost & in_img, 1], s=90, facecolors="none", edgecolors="red", lw=1.2,
              label=f"{int((lost & in_img).sum())} without a star, {int((~in_img).sum())} off-image")
    for i, j in zip(n_in, p_in):
        a.plot([P[i, 0], Q[j, 0]], [P[i, 1], Q[j, 1]], color="lime", lw=0.8)
    a.set_xlim(-50, SKY_W + 50)
    a.set_ylim(SKY_H + 50, -50)
    a.legend(fontsize=7, loc="lower left")
    a.set_title(f"named '{fit.name}', {scale:.0f} px/unit, mean residual {dd.mean() if len(dd) else 0:.1f}px", fontsize=9)

    a = fig.add_subplot(gs[1])
    art = cv2.imread(os.path.join(DATA, "patterns", f"{pattern.name}_pattern.png"), cv2.IMREAD_UNCHANGED)
    if art.ndim == 3 and art.shape[2] == 4:
        alpha = art[:, :, 3:4].astype(float) / 255
        art = (art[:, :, :3] * alpha + 40 * (1 - alpha)).astype(np.uint8)
    a.imshow(cv2.cvtColor(art, cv2.COLOR_BGR2RGB))
    a.scatter(pattern.raw[n_in, 0], pattern.raw[n_in, 1], s=70, facecolors="none", edgecolors="lime", lw=1.5)
    a.scatter(pattern.raw[lost, 0], pattern.raw[lost, 1], s=70, facecolors="none", edgecolors="red", lw=1.2)
    a.set_title(f"{pattern.name}: {len(pattern.nodes)} nodes, {len(pattern.pairs)} seed pairs", fontsize=9)
    a.set_xticks([])
    a.set_yticks([])

    a = fig.add_subplot(gs[2])
    top = outcome.ranking[:5][::-1]
    a.barh([r[0] for r in top], [r[1] for r in top], color=["tab:green" if r[0] == fit.name else "0.6" for r in top])
    for y, r in enumerate(top):
        a.text(max(r[1], 0) + 0.3, y, f"raw {r[2]:.1f} - null {r[3]:.1f}+{r[4]:.1f}  {r[5]}/{r[6]}", va="center", fontsize=7)
    a.set_xlim(0, max(r[1] for r in top) * 1.6 + 1)
    a.set_xlabel("calibrated score")
    a.set_title("top 5", fontsize=9)
    plt.tight_layout()
    plt.show()
    print(f"plot E: {time.time() - _t:.1f}s")

## 15. Results on the labelled training scenes

Solves the three labelled scenes and prints, per scene, the metric components, the top 5 names with their raw and null scores, how well the true figure fitted, and a per-patch table. Passing `tune=True` additionally grid-searches the presence thresholds; it is off by default because 116 labelled patches are too few to tune on safely.

In [ ]:
if IN_NOTEBOOK:
    train_outcomes, _ = report_training(DATA, pattern_bank, settings, show_tiles=True, tune=False)

## 16. Validation scenes → `submission.csv`

Scenes are solved in the order of `sample_submission.csv` and the file keeps that layout (`-1` or `"(x, y, m)"` per patch, then the name). Since a wrong name costs no more than `unknown`, a name is always given. The submitted run uses search stride 2, which cuts the sweep cost by four at a small loss of recall.

In [ ]:
settings = configure(search_stride=2, mem_budget_gb=4.0, search_batch=6, corr_batch=16384)
pattern_bank = load_patterns(DATA, settings)
print("device:", DEVICE, "| search stride:", settings.search_stride, "| figures:", len(pattern_bank))

In [ ]:
OUT_DIR = "/content" if IN_COLAB else "."


def download_if_colab(path):
    if IN_COLAB:
        from google.colab import files
        files.download(path)


def summarise_submission(path):
    import pandas as pd
    table = pd.read_csv(path)
    print(table.shape)
    print(table["constellation"].value_counts())


if IN_NOTEBOOK:
    val_outcomes = predict_validation(DATA, pattern_bank, settings, out_path=os.path.join(OUT_DIR, "submission.csv"))
    summarise_submission(os.path.join(OUT_DIR, "submission.csv"))
    download_if_colab(os.path.join(OUT_DIR, "submission.csv"))

## 17. Follow-up experiments

Each experiment applies one scene-agnostic rule uniformly to all validation scenes.

- **(a) How stable is the name?** Scenes with a weak name (lead below 7 or score below 3) are re-named with 300 and with 500 bright stars, reusing the stored localisation. Diagnostic only.
- **(b) Full resolution where it matters.** The weak scenes are solved again at stride 1. The new answer replaces the old one if it keeps the same name, or if it changes the name with a lead at least 2 larger and a score at least as high.
- **(c) Wider candidate funnel.** A complete re-run with 8 peaks per template, 1,800 sweep candidates and a 180-candidate screen.

In [ ]:
def score_gap(ranked):
    return ranked[0].score - ranked[1].score if len(ranked) >= 2 else 999.0


def is_weak(o: SceneOutcome):
    return score_gap(o.ranked) < 7.0 or o.fit.score < 3.0


def reidentify(o: SceneOutcome, cfg2):
    """Name the scene again from its stored localisation, with other naming settings."""
    Q, qflux, qsat, qmag, keep, bright = identification_points(o.stars, o.before, cfg2)
    return identify_constellation(Q[keep], qmag[keep], pattern_bank, cfg2, bright_xy=bright)


if IN_NOTEBOOK:
    variants = (("300 stars", configure(settings, id_n_stars=300)),
                ("500 stars", configure(settings, id_n_stars=500, id_n_complete=800)))
    print("=" * 80)
    for o in val_outcomes:
        print(f"{o.sid:18s} {o.fit.name:20s} score={o.fit.score:7.2f} lead={score_gap(o.ranked):6.2f} "
              f"runner={o.ranked[1].name if len(o.ranked) > 1 else '-'}")
    weak = [o for o in val_outcomes if is_weak(o)]
    print("weak:", [o.sid for o in weak])
    NAME_SWEEP, _t = {}, time.time()
    for n, o in enumerate(weak, 1):
        print(f"\n[{n}/{len(weak)}] {o.sid}\n  default: {o.fit.name:20s} score={o.fit.score:7.2f} lead={score_gap(o.ranked):6.2f}")
        NAME_SWEEP[o.sid] = {"default": (o.fit, o.ranked, score_gap(o.ranked))}
        for label, cfg2 in variants:
            f2, r2 = reidentify(o, cfg2)
            NAME_SWEEP[o.sid][label] = (f2, r2, score_gap(r2))
            print(f"  {label}: {f2.name:20s} score={f2.score:7.2f} lead={score_gap(r2):6.2f}" + (f" runner={r2[1].name}" if len(r2) > 1 else ""))
    print(f"\nnaming sweep: {(time.time() - _t) / 60:.2f} min")

In [ ]:
if IN_NOTEBOOK:
    full = configure(settings, search_stride=1, mem_budget_gb=4.0, search_batch=6, corr_batch=16384)
    patterns_full = load_patterns(DATA, full)
    adaptive, _t = [], time.time()
    for o in val_outcomes:
        g = score_gap(o.ranked)
        if not is_weak(o):
            print(f"[{o.sid}] keep stride-2 result: {o.fit.name} score={o.fit.score:.2f} lead={g:.2f}")
            adaptive.append(o)
            continue
        print(f"\n[{o.sid}] re-running at full resolution (was {o.fit.name}, score={o.fit.score:.2f}, lead={g:.2f})")
        started = time.time()
        hires = solve_scene(prepare_scene(*load_scene(os.path.join(DATA, "validation", o.sid)), full), patterns_full, full, tag=o.sid + "_full")
        g2 = score_gap(hires.ranked)
        print(f"[{o.sid}] {o.fit.name} ({o.fit.score:.2f}, lead {g:.2f}) -> {hires.fit.name} ({hires.fit.score:.2f}, lead {g2:.2f}) "
              f"| {(time.time() - started) / 60:.1f} min")
        if hires.fit.name == o.fit.name or (g2 >= g + 2.0 and hires.fit.score >= o.fit.score):
            print(f"[{o.sid}] using the full-resolution result")
            adaptive.append(replace(hires, sid=o.sid))
        else:
            print(f"[{o.sid}] name change not convincing: keeping stride 2")
            adaptive.append(o)
    print(f"\nadaptive pass: {(time.time() - _t) / 60:.2f} min")
    write_submission(adaptive, DATA, os.path.join(OUT_DIR, "submission_adaptive_hires.csv"))
    download_if_colab(os.path.join(OUT_DIR, "submission_adaptive_hires.csv"))

In [ ]:
if IN_NOTEBOOK:
    wide = configure(settings, search_stride=2, search_peaks_each=8, search_keep=1800, screen_keep=180,
                     mem_budget_gb=4.0, search_batch=6, corr_batch=16384)
    print("wide recall:", wide.search_peaks_each, "peaks/template,", wide.search_keep, "candidates,", wide.screen_keep, "screened")
    _t = time.time()
    wide_outcomes = predict_validation(DATA, load_patterns(DATA, wide), wide, out_path=os.path.join(OUT_DIR, "submission_wide.csv"))
    summarise_submission(os.path.join(OUT_DIR, "submission_wide.csv"))
    print(f"runtime: {(time.time() - _t) / 60:.2f} min")
    download_if_colab(os.path.join(OUT_DIR, "submission_wide.csv"))

## 18. Known weaknesses

- **The bright-patch prior cuts both ways.** Calling every bright-star patch present earns localisation and geometric credit, but every absent bright patch becomes a false positive.
- **Blown-out patches** carry no structure to correlate, so they rely entirely on placement onto the named figure. A bright patch whose star is not among the 500 brightest can never be found.
- **Tiny figures** (5 nodes or fewer) are hard to name from 350 stars: random subsets fit them completely, and their scrambled null scores as well as the real figure.
- **The completeness penalty** assumes that every in-image figure star is among the 600 brightest detections.
- **The zoom prior** of 0.7 to 1.45 would miss patches scaled outside that range.
- **Few labels.** The presence thresholds and bright-patch settings (60 refined candidates, 6 shift searches) were checked on only three labelled scenes.

## 19. Mapping to course topics

| where in the pipeline | course topic |
|---|---|
| pixel histograms and CDFs (diagnostic A) | intensity distributions, lookup tables |
| background model | box-filter downsampling, median filtering of outliers, piecewise-flat prior |
| image borders | padding, reflect-101 borders |
| saturated-pixel counts | summed-area tables (integral images) |
| `sign(x)·sqrt(|x|)` stretch | point-wise intensity transforms |
| smoothing | separable Gaussian filtering |
| halo removal | subtracting an image's own radial profile |
| star detection (diagnostic B) | matched filtering by cross-correlation, LoG / DoG blob detection |
| peak picking | non-maximum suppression via dilation |
| centroids | box sums as sliding dot products |
| patch descriptor | detect → describe → match; rotation and scale invariance |
| drawings to point sets | thresholding and normalisation |
| whole-sky search (diagnostic C) | template matching over rotations, `TM_CCOEFF_NORMED` |
| neighbour votes | brute-force keypoint comparison |
| coarse-to-fine search | refinement over position and scale |
| presence rule (diagnostic D) | ratio test, double thresholding |
| naming (diagnostic E) | solving `p1 = T·p2` with outliers, RANSAC-style verification |
| placement | search restricted to keypoints |
| beyond the lectures | MAD noise scale, random-pose null, likelihood-ratio scoring, scrambled-figure null, connected components, distance transform |

In [ ]:
if __name__ == "__main__" and not IN_NOTEBOOK:
    import argparse
    cli = argparse.ArgumentParser(description="Constellation Detection (CS-GY 6643 Project 1, Q5)")
    cli.add_argument("--data", default=None, help="participant/ folder (default: auto-detect)")
    cli.add_argument("--train-eval", action="store_true", help="score the labelled training scenes")
    cli.add_argument("--predict", action="store_true", help="run the validation scenes and write the CSV")
    cli.add_argument("--out", default="submission.csv")
    cli.add_argument("--stride", type=int, default=None, help="search stride (default 1 on GPU, 2 on CPU)")
    args = cli.parse_args()
    if args.data:
        DATA = os.path.abspath(args.data)
    assert DATA and os.path.isdir(os.path.join(DATA, "patterns")), "participant/ folder not found: pass --data"
    settings = configure(search_stride=args.stride or (1 if DEVICE.type == "cuda" else 2), mem_budget_gb=4.0, search_batch=6, corr_batch=16384)
    pattern_bank = load_patterns(DATA, settings)
    if args.train_eval:
        report_training(DATA, pattern_bank, settings, show_tiles=True, tune=False)
    if args.predict or not args.train_eval:
        predict_validation(DATA, pattern_bank, settings, out_path=args.out)